# PlantDoc SpecialistRoute — Main Pipeline (protocol v2)

**How to use**
1. Run sections **1–7** (setup, config, definitions). They only define things — nothing trains.
2. Run sections **8–9a** (more definitions), then **9b** to train everything: all 8 components for seed 42
   (Classifier first), the seed-42 pipeline evaluation, then the next seed — up to all 5 seeds.
3. If anything fails or the kernel dies, just run 1–9a again and then 9b: finished runs are skipped.
4. Section **10** is optional: one component by hand, or re-evaluating chosen seeds.

**Rules**
- All settings live in section 2. `apply_component()` resets `CONFIG` before every run, so nothing leaks between runs.
- For quick tests use overrides, e.g. `apply_component("Tomato", epochs=5)` — never edit `CONFIG` by hand.
- Do **not** run cells from `PlantDoc_Archive_Tools.ipynb` in this kernel: several of them redefine functions used here.

**Protocol v2**: final dataset from `prepare_final_dataset` (official 236-image test set, leakage removed, 10% held-out validation);
checkpoint selection on `val_loss` of the held-out validation set; test used only for final evaluation.


## 1. Setup

In [ ]:
import os
os.environ["KERAS_BACKEND"] = "tensorflow"  # Force TensorFlow backend


import keras_cv
import copy, gc, hashlib, json, shutil
import os
import math
import numpy as np
import tensorflow as tf

# ── GPU: allocate memory as needed instead of grabbing it all up front (reduces OOM) ──
for _gpu in tf.config.list_physical_devices("GPU"):
    try:
        tf.config.experimental.set_memory_growth(_gpu, True)
    except RuntimeError as _e:          # only possible before the GPU is first used
        print(f"[GPU] memory growth not set ({_e}) — restart the kernel and run this cell first")
import matplotlib.pyplot as plt
import seaborn as sns
import random
from sklearn.model_selection import KFold
from sklearn.metrics import classification_report, confusion_matrix
from tensorflow.keras import layers, Model
from tensorflow.keras.applications import MobileNetV2, EfficientNetB0, EfficientNetB3, EfficientNetV2S, InceptionResNetV2, EfficientNetV2L, ConvNeXtBase, ConvNeXtLarge, NASNetLarge, EfficientNetB7, ConvNeXtTiny, ConvNeXtSmall
from tensorflow.keras.applications.nasnet  import preprocess_input as nasnet_preprocess
from tensorflow.keras.applications import mobilenet_v2, efficientnet
from tensorflow.keras.applications.inception_resnet_v2 import preprocess_input as inceptionresnetv2_preprocess

os.environ['PYTHONHASHSEED'] = '42'
random.seed(42)
np.random.seed(42)
tf.random.set_seed(42)


print(f"TensorFlow version: {tf.__version__}")
print(f"GPUs available: {len(tf.config.list_physical_devices('GPU'))}")

## 2. Configuration
Model weight paths, then `CONFIG_BASE` (shared settings) and `COMPONENTS` (per-component presets).

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# MODEL PATHS — centralized registry
# Every trained/pretrained model path referenced anywhere in this notebook lives
# here. Update a path in exactly one place; every cell that uses it (CONFIG plus
# every standalone evaluation cell below) reads from this dict rather than
# hardcoding its own copy of the same path.
# ═══════════════════════════════════════════════════════════════════════════════

MODEL_PATHS = {
    # ── Pretrained weights (not a PlantDoc-trained checkpoint) ────────────────
    "noisy_student_pretrained_weights": "/home/auc/PlantDoc/Models/efficientnet-b0_noisy-student_notop.h5",

    # ── PlantVillage-trained models (used both for CONFIG's transfer-learning
    #    path and for the standalone zero-shot generalization eval cell — same
    #    underlying file, referenced from two different places) ────────────────
    "plantvillage_potato_model": "./plantdoc_local/working/plantdoc_experiments/exp_70_plantvillage_Potato/noisy.keras",
    "plantvillage_grape_model":  "./plantdoc_local/working/plantdoc_experiments/exp_71_plantvillage_Grape/convnextbase.keras",

    # ── Trained PlantDoc specialist checkpoints, used by name elsewhere ────────
    "rest_specialist_model": "./plantdoc_local/working/plantdoc_experiments/exp_47_plantDoc_cleaned_rest_v2/convnextbase.keras",

    # ── Tomato F1-weighted / standalone ensemble (two models compared together) ─
    "tomato_ensemble_convnextbase":  "./plantdoc_local/working/plantdoc_experiments/exp_46_plantDoc_cleaned_tomato_v6/convnextbase.keras",
    "tomato_ensemble_convnextlarge": "./plantdoc_local/working/plantdoc_experiments/exp_59_plantDoc_cleaned_tomato_v11/ConvNeXtLarge.keras",


    # ── ImageNet-pretrained backbone weights (no-top, .h5) ─────────────────────
    # NOTE: weights="imagenet" (the Keras default) auto-DOWNLOADS these from a
    # remote server on first use and caches them in ~/.keras/models/ — this
    # requires internet access. The entries below point to local files instead,
    # for a fully offline-capable setup. If you DO have internet access and
    # don't mind the automatic download/cache behavior, you can set any of
    # these entries back to the string "imagenet" instead of a local path.
    "mobilenetv2_imagenet_weights":       "./plantdoc_local/input/imagenet_weights/mobilenetv2_notop.h5",
    "efficientnetb0_imagenet_weights":    "./plantdoc_local/input/imagenet_weights/efficientnetb0_notop.h5",
    "efficientnetb3_imagenet_weights":    "./plantdoc_local/input/imagenet_weights/efficientnetb3_notop.h5",
    "efficientnetv2s_imagenet_weights":   "./plantdoc_local/input/imagenet_weights/efficientnetv2s_notop.h5",
    "efficientnetv2l_imagenet_weights":   "./plantdoc_local/input/imagenet_weights/efficientnetv2l_notop.h5",
    "inceptionresnetv2_imagenet_weights": "./plantdoc_local/input/imagenet_weights/inceptionresnetv2_notop.h5",
    "convnextbase_imagenet_weights":      "/home/auc/PlantDoc/Models/convnext_base_notop.h5",
    "convnextlarge_imagenet_weights":     "/home/auc/PlantDoc/Models/convnext_large_notop.h5",
    "convnexttiny_imagenet_weights":      "./plantdoc_local/input/imagenet_weights/convnexttiny_notop.h5",
    "convnextsmall_imagenet_weights":     "./plantdoc_local/input/imagenet_weights/convnextsmall_notop.h5",
    "nasnetlarge_imagenet_weights":       "./plantdoc_local/input/imagenet_weights/nasnetlarge_notop.h5",

    # ── Single-purpose evaluation-cell models ──────────────────────────────────
    "apple_copypaste_test_model":       "./plantdoc_local/working/plantdoc_experiments/exp_88_plantDoc_CopyPasteAug_Apple/ConvNeXtLarge.keras",
    "classifier_tta_eval_model":        "./plantdoc_local/working/plantdoc_experiments/exp_66_cleaned_classifier_v3/ConvNeXtLarge.keras",
    "apple_tta_eval_model":             "./plantdoc_local/working/plantdoc_experiments/exp_27_plantdoc_apple/convnextbase.keras",
}

print(f"[MODEL_PATHS] {len(MODEL_PATHS)} model paths registered.")


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# 2. CONFIGURATION — the only cell you normally edit.
#
#   CONFIG_BASE : settings shared by every run (paths, protocol, training defaults)
#   COMPONENTS  : per-component presets (model, image size, epochs, LR, fine-tuning, classes)
#   apply_component(name, **overrides) : RESETS CONFIG to CONFIG_BASE, then applies the
#       preset — so nothing can leak from a previous run in the same kernel session.
#
# Never edit CONFIG directly between runs; change CONFIG_BASE / COMPONENTS, or pass
# overrides, e.g. apply_component("Tomato", epochs=10) for a quick smoke test.
# ═══════════════════════════════════════════════════════════════════════════════
import copy

CONFIG_BASE = {
    # ─── Paths ────────────────────────────────────────────────────────────────
    "dataset_root":     "/home/auc/PlantDoc/Datasets/dataset",     # final train/val/test (prepare_final_dataset)
    "derived_root":     "/home/auc/PlantDoc/Datasets/derived_v2",  # specialist + classifier folders (auto-built)
    "output_dir":       "/home/auc/PlantDoc/Experiments",
    "experiment_prefix": "v2",                                     # experiment folder = f"{prefix}_{component}"

    # ─── Protocol ─────────────────────────────────────────────────────────────
    "checkpoint_monitor":  "val_loss",      # selection on the HELD-OUT validation set
    "expected_test_count": 236,
    "multi_seed_list":     [42, 123, 2024, 779, 1027],

    # ─── Dataset / loader ─────────────────────────────────────────────────────
    "img_size":         384,      # set per component by COMPONENTS
    "num_classes":      None,     # auto-set from the class folders
    "data_fraction":    1.0,
    "val_split":        0.0,      # validation comes from its own folder
    "random_seed":      42,
    "batch_size":       32,

    # ─── Training (defaults; COMPONENTS override the per-component ones) ──────
    "epochs":             100,
    "learning_rate":      1e-4,
    "fine_tune":          False,
    "fine_tune_lr":       5e-5,
    "fine_tune_at_epoch": 20,
    "reduce_lr_factor":   0.5,
    "reduce_lr_patience": 30,
    "reduce_lr_min_lr":   1e-6,
    "early_stopping":     False,
    "early_stopping_patience": 15,
    "use_cyclical_lr":    False,
    "cyclical_first_decay_steps": 15,
    "cyclical_t_mul":     1.1,
    "cyclical_m_mul":     1,
    "use_focal_loss":     False,
    "focal_loss_gamma":   2.0,
    "verbose":            1,

    # ─── Online augmentation ──────────────────────────────────────────────────
    "use_augmentation":   True,
    "use_cutmix":         True,
    "use_mixup":          True,
    "cutmix_alpha":       1.0,
    "mixup_alpha":        0.2,
    "use_randaugment":    False,
    "randaugment_num_ops": 2,
    "randaugment_magnitude": 0.3,
    "use_gridmask":       False,
    "gridmask_probability": 0.5,
    "gridmask_d_min":     40,
    "gridmask_d_max":     160,
    "gridmask_ratio":     0.5,
    "use_random_filter_augment": False,
    "random_filter_probability": 0.3,
    "use_copy_paste":     False,          # legacy ONLINE copy-paste (Copy-Paste V2 is offline — see archive)
    "copy_paste_probability": 0.3,
    "copy_paste_patch_ratio": 0.15,
    "healthy_class_index": 0,

    # ─── Preprocessing (images are already CLAHE-processed on disk) ───────────
    "use_clahe":          False,
    "clahe_clip_limit":   2.0,
    "clahe_tile_grid":    (8, 8),
    "use_offline_aug":    False,
    "offline_aug_dir":    "",

    # ─── Models ───────────────────────────────────────────────────────────────
    "models_to_train":    None,            # set by COMPONENTS
    "noisy_student_weights": MODEL_PATHS["noisy_student_pretrained_weights"],
    "plantvillage_model_path": MODEL_PATHS["plantvillage_potato_model"],
    "rest_model_path":    MODEL_PATHS["rest_specialist_model"],
    "mobilenet_head":     {"dropout": 0.3, "dense_units": 128},
    "efficientnet_head":  {"dropout": 0.4, "dense_units": 256},

    # ─── Saving ───────────────────────────────────────────────────────────────
    "save_model":         True,            # required: the best checkpoint is reloaded for evaluation
    "save_format":        ".keras",

    # ─── Set by apply_component() — do not edit ───────────────────────────────
    "component":          None,
    "experiment_name":    None,
    "dataset_dir_train":  None,
    "dataset_dir_val":    None,
    "dataset_dir_test":   None,
    "filter_by_plant":    False,
    "plant_keyword":      None,
}

# Per-component presets (values from the final v1 configs: Plantdoc.csv, exp_66 classifier,
# exp_122 Rest, potato v9). "keywords" select that component's class folders.
COMPONENTS = {
    "Classifier":  {"type": "classifier", "include_rest": True,
                    "model": "ConvNeXtLarge", "img_size": 384, "epochs": 140, "learning_rate": 7e-5,
                    "fine_tune": True, "fine_tune_lr": 5e-5, "fine_tune_at_epoch": 50},
    "Apple":       {"type": "specialist", "keywords": ["Apple"],
                    "model": "ConvNeXtLarge", "img_size": 384, "epochs": 100, "learning_rate": 1e-4,
                    "fine_tune": False},
    "Bell_pepper": {"type": "specialist", "keywords": ["Bell_pepper"],
                    "model": "convnextbase", "img_size": 384, "epochs": 140, "learning_rate": 1e-4,
                    "fine_tune": True, "fine_tune_lr": 5e-5, "fine_tune_at_epoch": 40},
    "Corn":        {"type": "specialist", "keywords": ["Corn"],
                    "model": "convnextbase", "img_size": 384, "epochs": 100, "learning_rate": 1e-4,
                    "fine_tune": True, "fine_tune_lr": 5e-5, "fine_tune_at_epoch": 40},
    "Potato":      {"type": "specialist", "keywords": ["Potato"],      # was noisy (EfficientNet-B0): its weights did not
                    "model": "convnextbase", "img_size": 384, "epochs": 100,  # load under Keras 3 -> same preset as Corn
                    "learning_rate": 1e-4, "fine_tune": True, "fine_tune_lr": 5e-5, "fine_tune_at_epoch": 40},
    "Tomato":      {"type": "specialist", "keywords": ["Tomato"],
                    "model": "ConvNeXtLarge", "img_size": 384, "epochs": 300, "learning_rate": 1e-4,
                    "fine_tune": False},
    "Grape":       {"type": "specialist", "keywords": ["Grape"],
                    "model": "convnextbase", "img_size": 384, "epochs": 50, "learning_rate": 1e-4,
                    "fine_tune": True, "fine_tune_lr": 5e-5, "fine_tune_at_epoch": 20},
    "Rest":        {"type": "rest", "keywords": ["Blueberry", "Cherry", "Peach", "Raspberry",
                                                 "Soyabean", "Squash", "Strawberry"],
                    "model": "ConvNeXtLarge", "img_size": 384, "epochs": 200, "learning_rate": 5e-5,
                    "fine_tune": False},
}
_PRESET_TRAINING_KEYS = ("img_size", "epochs", "learning_rate", "fine_tune", "fine_tune_lr", "fine_tune_at_epoch")

CONFIG = copy.deepcopy(CONFIG_BASE)


def final_split_dirs():
    return {s: os.path.join(CONFIG_BASE["dataset_root"], s) for s in ("train", "val", "test")}


def apply_component(name, **overrides):
    """Reset CONFIG to CONFIG_BASE, apply the preset for `name`, then any overrides."""
    if name not in COMPONENTS:
        raise KeyError(f"Unknown component '{name}'. Choose from {list(COMPONENTS)}")
    preset = COMPONENTS[name]
    CONFIG.clear()
    CONFIG.update(copy.deepcopy(CONFIG_BASE))
    dirs = final_split_dirs()
    CONFIG.update(dataset_dir_train=dirs["train"], dataset_dir_val=dirs["val"], dataset_dir_test=dirs["test"],
                  component=name, experiment_name=f"{CONFIG_BASE['experiment_prefix']}_{name}",
                  models_to_train=preset["model"])
    for k in _PRESET_TRAINING_KEYS:
        if k in preset:
            CONFIG[k] = preset[k]
    if preset["type"] == "classifier":
        d = build_classifier_dataset_v2(include_rest=preset["include_rest"])
        CONFIG.update(dataset_dir_train=d["train"], dataset_dir_val=d["val"], dataset_dir_test=d["test"],
                      filter_by_plant=False, plant_keyword=None)
    else:
        CONFIG.update(filter_by_plant=True, plant_keyword=list(preset["keywords"]))
    for k, v in overrides.items():
        if k not in CONFIG:
            raise KeyError(f"Override '{k}' is not a CONFIG key (typo?)")
        CONFIG[k] = v
    print(f"[Config] component={name} | model={CONFIG['models_to_train']} | img={CONFIG['img_size']} | "
          f"epochs={CONFIG['epochs']} | lr={CONFIG['learning_rate']} | fine_tune={CONFIG['fine_tune']} | "
          f"experiment={CONFIG['experiment_name']}" + (f" | overrides={overrides}" if overrides else ""))
    return CONFIG


def get_model_save_path(model_name: str):
    """
    Returns the .keras save path for a model, OR None if CONFIG["save_model"]
    is False — callers (get_callbacks) must check for None and skip
    ModelCheckpoint entirely in that case. Added to support multi-seed sweeps
    where only metrics matter and writing N checkpoint files is wasteful.
    """
    if not CONFIG.get("save_model", True):
        return None
    out = os.path.join(
        CONFIG["output_dir"],
        CONFIG["experiment_name"],
        f"{model_name}{CONFIG['save_format']}"
    )
    os.makedirs(os.path.dirname(out), exist_ok=True)
    if os.path.exists(out):
        raise FileExistsError(f"Model already exists: {out}\nChange experiment_name in CONFIG.")
    return out


def print_config():
    print("\n========== PlantDoc Config ==========")
    for k, v in CONFIG.items():
        print(f"  {k}: {v}")
    print("======================================\n")


print(f"[Config] {len(COMPONENTS)} components: {list(COMPONENTS)}")


## 3. Data
Held-out validation, stale-proof specialist/classifier folders, protocol check.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# 3. DATA — held-out validation, test only for evaluation, stale-proof folders
#   train / val come from the training data; test is loaded ONLY by get_test_dataset().
#   Specialist and classifier folders are built under CONFIG["derived_root"] and
#   verified against their source on every use — a stale folder raises an error.
# ═══════════════════════════════════════════════════════════════════════════════
TF_IMAGE_EXT = (".bmp", ".gif", ".jpeg", ".jpg", ".png")   # what image_dataset_from_directory reads


def _class_dirs(root):
    return sorted(d for d in os.listdir(root) if os.path.isdir(os.path.join(root, d)))


def _file_counts(root):
    """{class: (files_on_disk, files_tf_will_read)}"""
    out = {}
    for c in _class_dirs(root):
        files = [f for f in os.listdir(os.path.join(root, c)) if not f.startswith(".")]
        out[c] = (len(files), sum(f.lower().endswith(TF_IMAGE_EXT) for f in files))
    return out


def _matches(cls, keywords):
    return any(kw.lower() in cls.lower() for kw in keywords)


def _filtered_split_dirs(keywords, src=None):
    """Build/verify derived_root/filtered_<keywords>__<hash>/{train,val,test}."""
    if isinstance(keywords, str):
        keywords = [keywords]
    src = src or {s: CONFIG[f"dataset_dir_{s}"] for s in ("train", "val", "test")}
    tag = hashlib.md5("|".join(src[s] for s in ("train", "val", "test")).encode()).hexdigest()[:8]
    base = os.path.join(CONFIG_BASE["derived_root"], f"filtered_{'_'.join(keywords)}__{tag}")
    dst = {s: os.path.join(base, s) for s in src}
    for s in ("train", "val", "test"):
        classes = [c for c in _class_dirs(src[s]) if _matches(c, keywords)]
        if not classes:
            raise RuntimeError(f"No class folder in {src[s]} matches {keywords}.")
        expected = {c: len(os.listdir(os.path.join(src[s], c))) for c in classes}
        if os.path.exists(dst[s]):
            found = {c: len(os.listdir(os.path.join(dst[s], c))) for c in _class_dirs(dst[s])}
            if found != expected:
                raise RuntimeError(f"STALE folder {dst[s]}: {found} != source {expected}. Delete {base} and re-run.")
            continue
        os.makedirs(dst[s])
        for c in classes:
            shutil.copytree(os.path.join(src[s], c), os.path.join(dst[s], c))
        print(f"  [Filter] {s}: {len(classes)} classes, {sum(expected.values())} images -> {dst[s]}")
    return dst


def build_filtered_dataset(keywords):
    d = _filtered_split_dirs(keywords)
    return d["train"], d["test"]


def _split_dirs():
    if CONFIG.get("filter_by_plant") and CONFIG.get("plant_keyword"):
        return _filtered_split_dirs(CONFIG["plant_keyword"])
    return {s: CONFIG[f"dataset_dir_{s}"] for s in ("train", "val", "test")}


def _check_same_classes(dirs):
    names = {s: _class_dirs(d) for s, d in dirs.items()}
    if len({tuple(v) for v in names.values()}) != 1:
        raise RuntimeError(f"Class folders differ between splits: {names}")
    return names["train"]


def _load_dir(path, shuffle):
    return tf.keras.utils.image_dataset_from_directory(
        path, seed=CONFIG["random_seed"], image_size=(CONFIG["img_size"], CONFIG["img_size"]),
        batch_size=None, label_mode="int", shuffle=shuffle)


def _build_raw_datasets(data_dir: str = None):
    """Returns (train_raw, val_raw, class_names). val_raw is the HELD-OUT validation set."""
    dirs = _split_dirs()
    expected = _check_same_classes(dirs)
    train_ds_raw = _load_dir(dirs["train"], shuffle=True)
    val_ds_raw = _load_dir(dirs["val"], shuffle=False)
    class_names = train_ds_raw.class_names
    if class_names != expected or val_ds_raw.class_names != class_names:
        raise RuntimeError(f"Class order mismatch: train {class_names} vs val {val_ds_raw.class_names}")
    print(f"[DataLoader] {len(class_names)} classes | train: {dirs['train']} | val: {dirs['val']}")
    CONFIG["num_classes"] = len(class_names)
    fraction = CONFIG.get("data_fraction", 1.0)
    if fraction < 1.0:
        n_train = sum(1 for _ in train_ds_raw)
        n_keep = max(1, int(n_train * fraction))
        train_ds_raw = train_ds_raw.take(n_keep)
        print(f"[DataLoader] data_fraction={fraction}: using {n_keep}/{n_train} samples.")
    return train_ds_raw, val_ds_raw, class_names


def get_test_dataset(model_name: str):
    """Preprocessed TEST set (no augmentation) + class names. Final evaluation only."""
    dirs = _split_dirs()
    expected = _check_same_classes(dirs)
    test_raw = _load_dir(dirs["test"], shuffle=False)
    if test_raw.class_names != expected:
        raise RuntimeError(f"Test class order {test_raw.class_names} != train {expected}")
    return _apply_preprocessing(test_raw, model_name, CONFIG["batch_size"], augment=False), test_raw.class_names


def _train_class_counts():
    """Image count per class in the current training folder (class order = loader order)."""
    return [v[1] for _, v in sorted(_file_counts(_split_dirs()["train"]).items())]


def species_of(class_folder):
    """Component that owns a class folder: a specialist name, or 'Rest'."""
    for name, p in COMPONENTS.items():
        if p["type"] == "specialist" and _matches(class_folder, p["keywords"]):
            return name
    return "Rest"


def build_classifier_dataset_v2(include_rest=True, rebuild=False):
    """Species-level classifier data (train/val/test) from the final dataset."""
    out = os.path.join(CONFIG_BASE["derived_root"],
                       "crop_classifier_7class" if include_rest else "crop_classifier_6class")
    if os.path.exists(out) and rebuild:
        shutil.rmtree(out)
    complete = os.path.exists(os.path.join(out, "_complete"))
    for split, src in final_split_dirs().items():
        dst = os.path.join(out, split)
        expected = {}
        for cls in _class_dirs(src):
            species = species_of(cls)
            if species == "Rest" and not include_rest:
                continue
            files = [f for f in os.listdir(os.path.join(src, cls)) if not f.startswith(".")]
            expected[species] = expected.get(species, 0) + len(files)
            if not complete:
                os.makedirs(os.path.join(dst, species), exist_ok=True)
                for f in files:
                    shutil.copy(os.path.join(src, cls, f), os.path.join(dst, species, f"{cls}__{f}"))
        found = {c: len(os.listdir(os.path.join(dst, c))) for c in _class_dirs(dst)}
        if found != expected:
            raise RuntimeError(f"[Classifier] {dst} does not match its source: {found} vs {expected}. "
                               "Call build_classifier_dataset_v2(rebuild=True).")
    open(os.path.join(out, "_complete"), "w").close()
    return {s: os.path.join(out, s) for s in ("train", "val", "test")}


def check_protocol():
    """Pre-flight check of the final dataset, component coverage and CONFIG."""
    ok = True
    print("=" * 70 + "\nPROTOCOL CHECK\n" + "=" * 70)
    dirs = final_split_dirs()
    totals, names, seen = {}, {}, {}
    for s, d in dirs.items():
        counts = _file_counts(d)
        names[s] = list(counts)
        disk, readable = sum(v[0] for v in counts.values()), sum(v[1] for v in counts.values())
        totals[s] = readable
        print(f"  {s:5s}: {readable} images ({disk} files), {len(counts)} classes")
        if disk != readable:
            ok = False
            print(f"    ! {disk - readable} file(s) have extensions the TF loader skips {TF_IMAGE_EXT}")
        for c in counts:
            for f in os.listdir(os.path.join(d, c)):
                with open(os.path.join(d, c, f), "rb") as fh:          # compare CONTENT, not names:
                    digest = hashlib.md5(fh.read()).hexdigest()         # Kaggle PlantDoc reuses filenames across splits
                seen.setdefault(digest, []).append(f"{s}:{c}/{f}")
    if len({tuple(v) for v in names.values()}) != 1:
        ok = False
        print("    ! class folders differ between train/val/test")
    if totals["test"] != CONFIG_BASE["expected_test_count"]:
        ok = False
        print(f"    ! test has {totals['test']} images, expected {CONFIG_BASE['expected_test_count']}")
    shared = [v for v in seen.values() if len({x.split(":")[0] for x in v}) > 1]
    if shared:
        ok = False
        print(f"    ! {len(shared)} identical image(s) appear in more than one split (leakage), e.g. {shared[:2]}")
    else:
        print("  no identical image in more than one split (content hash)")
    rest_kw = next(p["keywords"] for p in COMPONENTS.values() if p["type"] == "rest")
    for cls in names["train"]:
        owners = [n for n, p in COMPONENTS.items() if p["type"] in ("specialist", "rest") and _matches(cls, p["keywords"])]
        if len(owners) != 1:
            ok = False
            print(f"    ! class '{cls}' is matched by {owners or 'no component'} — fix COMPONENTS keywords")
    for key, want in [("use_clahe", False), ("save_model", True)]:
        if CONFIG.get(key) != want:
            ok = False
            print(f"    ! CONFIG['{key}'] = {CONFIG.get(key)} (must be {want})")
    if CONFIG.get("component"):
        print(f"  current component: {CONFIG['component']} | experiment: {CONFIG['experiment_name']}")
        print(f"  training data: {CONFIG['dataset_dir_train']}"
              + (f" filtered by {CONFIG['plant_keyword']}" if CONFIG.get("filter_by_plant") else ""))
    print(f"  checkpoint selection: {CONFIG_BASE['checkpoint_monitor']} on the held-out validation set")
    print("  RESULT: " + ("OK — ready to train" if ok else "FIX THE ITEMS MARKED ! BEFORE TRAINING"))
    return ok


def verify_labels(dataset, class_names: list, n_samples: int = 5):
    """Print n sample (label index, class name) pairs from the dataset."""
    print("\n[Label Verification] First few labels in dataset:")
    for images, labels in dataset.take(1):
        for i in range(min(n_samples, len(labels))):
            lbl = int(labels[i].numpy())
            print(f"  Sample {i}: label={lbl}  class='{class_names[lbl]}'")
    print()


def _get_dataset_directory() -> str:
    if CONFIG["use_offline_aug"]:
        aug_dir = CONFIG["offline_aug_dir"]
        if os.path.isdir(aug_dir):
            print(f"[DataLoader] Using offline-augmented dataset: {aug_dir}")
            return aug_dir
        else:
            print(f"[DataLoader] WARNING: offline_aug_dir  qnot found. Falling back.")
    return CONFIG["dataset_dir_train"]


print("Data functions defined.")


## 4. Preprocessing & augmentation

In [ ]:
import cv2
import numpy as np
import tensorflow as tf

# ── Filter functions (numpy/cv2 based, wrapped for use inside tf.py_function) ──

def _filter_sharpen(image_np):
    gaussian = cv2.GaussianBlur(image_np, (0, 0), 3)
    sharpened = cv2.addWeighted(image_np, 1.5, gaussian, -0.5, 0)
    return sharpened


def _filter_saturation_boost(image_np):
    hsv = cv2.cvtColor(image_np, cv2.COLOR_RGB2HSV).astype(np.float32)
    hsv[:, :, 1] = np.clip(hsv[:, :, 1] * 1.4, 0, 255)
    saturated = cv2.cvtColor(hsv.astype(np.uint8), cv2.COLOR_HSV2RGB)
    return saturated


def _filter_bilateral_denoise(image_np):
    denoised = cv2.bilateralFilter(image_np, d=9, sigmaColor=75, sigmaSpace=75)
    return denoised


_FILTER_FUNCTIONS = [_filter_sharpen, _filter_saturation_boost, _filter_bilateral_denoise]


def _apply_random_filter_numpy(image_np, choice):
    """
    image_np: numpy array, uint8, shape (H, W, 3)
    choice: int, which filter to apply (0 = sharpen, 1 = saturation, 2 = bilateral)
            choice == -1 means "no filter, return original unchanged"
    """
    image_np = image_np.astype(np.uint8)
    if choice == -1:
        return image_np
    return _FILTER_FUNCTIONS[choice](image_np)


def random_filter_augment(image, probability=0.3):
    """
    TF-graph-compatible random filter augmentation.
    With probability `probability`, applies one randomly chosen filter
    (sharpen, saturation boost, or bilateral denoise) to the image.
    Otherwise returns the image unchanged.

    image: tf.Tensor, uint8 or float32, shape (H, W, 3), values in [0, 255] range
            (apply this BEFORE model-specific preprocess_input normalization,
            same position as your existing online augmentation layers)
    """
    apply_filter = tf.random.uniform([]) < probability
    filter_choice = tf.random.uniform([], 0, 3, dtype=tf.int32)  # 0, 1, or 2

    def _apply():
        choice_value = tf.cast(filter_choice, tf.int64)
        filtered = tf.py_function(
            func=lambda img, c: _apply_random_filter_numpy(img.numpy(), int(c.numpy())),
            inp=[tf.cast(image, tf.uint8), choice_value],
            Tout=tf.uint8,
        )
        filtered.set_shape(image.shape)
        return tf.cast(filtered, image.dtype)

    def _no_apply():
        return image

    result = tf.cond(apply_filter, _apply, _no_apply)
    return result


# ═══════════════════════════════════════════════════════════════════════════════
# Integration into your existing training pipeline
# ═══════════════════════════════════════════════════════════════════════════════
#
# In your _apply_preprocessing function, add this call AFTER the standard
# aug_layer (flip/rotation/zoom/etc.) but BEFORE the model-specific preprocess_fn,
# and ONLY for the training set (not validation):
#
#   def process_train(image, label):
#       if aug_layer is not None:
#           image = aug_layer(image, training=True)
#       image = random_filter_augment(image, probability=0.3)   # ← add this line
#       image = preprocess_fn(image)
#       label = tf.one_hot(label, num_classes)
#       return image, label
#
# Add to CONFIG:
#   "use_random_filter_augment": True,
#   "random_filter_probability": 0.3,
#
# Then gate the call:
#   if CONFIG.get("use_random_filter_augment"):
#       image = random_filter_augment(image, probability=CONFIG.get("random_filter_probability", 0.3))
# ═══════════════════════════════════════════════════════════════════════════════

In [ ]:
def _make_preprocess_fn(model_name: str):
    """Returns the correct preprocessing function per backbone."""
    if model_name == "mobilenetv2":
        return mobilenet_v2.preprocess_input    # scales to [-1, 1]
    elif model_name == "efficientnetb0":
        return efficientnet.preprocess_input    # normalises [0,255] float
    elif model_name == "efficientnetb3":
        return efficientnet.preprocess_input  # same as B0
    elif model_name == "EfficientNetV2S":
        return tf.keras.applications.efficientnet_v2.preprocess_input
    elif model_name == "convnextbase":
        return tf.keras.applications.convnext.preprocess_input
    elif model_name == "efficientnetv2l":
        return tf.keras.applications.efficientnet_v2.preprocess_input
    elif model_name == "inceptionresnetv2":
        return inceptionresnetv2_preprocess
    elif model_name == "ConvNeXtLarge":
        return tf.keras.applications.convnext.preprocess_input
    elif model_name == "NASNetLarge":
        return nasnet_preprocess
    elif model_name == "noisy":
        return efficientnet.preprocess_input
    elif model_name == "DenseNet201":
        return densenet_preprocess
    elif model_name == "mobilevit":
        def mobilevit_preprocess(x):
            return tf.cast(x, tf.float32) / 255.0  # simple [0,1] normalization, no pretrained mean/std needed
        return mobilevit_preprocess
    elif model_name == "tomato_from_rest":
        return tf.keras.applications.convnext.preprocess_input
    elif model_name == "convnextsmall":
        return tf.keras.applications.convnext.preprocess_input
    elif model_name == "convnexttiny":
        return tf.keras.applications.convnext.preprocess_input
    elif model_name == "plantvillage_transfer":
        return efficientnet.preprocess_input  # adjust if backbone isn't ConvNeXt
    else:
        raise ValueError(f"Unknown model_name: {model_name}")
        
def build_randaugment_layer(value_range=(0, 255), augmentations_per_image=2, magnitude=0.3):
    return keras_cv.layers.RandAugment(
        value_range=value_range,
        augmentations_per_image=augmentations_per_image,
        magnitude=magnitude,
    )

# ═══════════════════════════════════════════════════════════════════════════════
# TRACK 3, TECHNIQUE 2: GridMask
# Masks a structured, evenly-spaced grid of regions across the image, forcing
# the model to rely on distributed features rather than one dominant region.
# Primary target: Corn (Gray Leaf Spot / Blight confusion).
# ═══════════════════════════════════════════════════════════════════════════════

def _gridmask_numpy(image_np, d_min=40, d_max=80, ratio=0.5, rotate_angle=0):
    """
    image_np: numpy array, shape (H, W, C), any dtype (mask is binary, applied as multiply)
    d_min/d_max: grid unit size range (pixels) — controls how large each masked
                 square and its gap are. Randomized per call for variety.
    ratio: fraction of each grid unit that is masked (kept black). 0.5 is a
           common default — half masked, half visible per grid cell.
    rotate_angle: optional rotation of the grid pattern in degrees (0 = no rotation,
                  aligned to image axes). Kept at 0 for simplicity/speed; can be
                  randomized later if axis-aligned masking proves too regular.
    """
    h, w = image_np.shape[0], image_np.shape[1]
    d = np.random.randint(d_min, d_max + 1)
    mask_len = int(d * ratio)

    mask = np.ones((h, w), dtype=np.float32)

    # Random offset so the grid doesn't always start at the same position
    start_x = np.random.randint(0, d)
    start_y = np.random.randint(0, d)

    for y in range(-d, h + d, d):
        for x in range(-d, w + d, d):
            y0 = y + start_y
            x0 = x + start_x
            y1 = min(max(y0, 0), h)
            y2 = min(max(y0 + mask_len, 0), h)
            x1 = min(max(x0, 0), w)
            x2 = min(max(x0 + mask_len, 0), w)
            mask[y1:y2, x1:x2] = 0.0

    mask = mask[:, :, np.newaxis]  # (H, W, 1) to broadcast across channels
    masked_image = (image_np.astype(np.float32) * mask).astype(image_np.dtype)
    return masked_image


def gridmask_augment(image, probability=0.5, d_min=40, d_max=80, ratio=0.5):
    """
    TF-graph-compatible GridMask augmentation.
    With probability `probability`, applies a structured grid mask to the image.
    Otherwise returns the image unchanged.

    image: tf.Tensor, uint8 or float32, shape (H, W, 3), values in [0, 255] range
           (apply this in the same position as your existing online augmentation,
           BEFORE model-specific preprocess_input normalization)
    """
    apply_mask = tf.random.uniform([]) < probability

    def _apply():
        masked = tf.py_function(
            func=lambda img: _gridmask_numpy(
                img.numpy().astype(np.uint8), d_min=d_min, d_max=d_max, ratio=ratio
            ),
            inp=[tf.cast(image, tf.uint8)],
            Tout=tf.uint8,
        )
        masked.set_shape(image.shape)
        return tf.cast(masked, image.dtype)

    def _no_apply():
        return image

    result = tf.cond(apply_mask, _apply, _no_apply)
    return result


# ═══════════════════════════════════════════════════════════════════════════════
# Integration into your existing process_train function
# ═══════════════════════════════════════════════════════════════════════════════
#
#   def process_train(image, label):
#       if aug_layer is not None:
#           image = aug_layer(image, training=True)
#       if CONFIG.get("use_gridmask"):
#           image = gridmask_augment(
#               image,
#               probability=CONFIG.get("gridmask_probability", 0.5),
#               d_min=CONFIG.get("gridmask_d_min", 40),
#               d_max=CONFIG.get("gridmask_d_max", 80),
#               ratio=CONFIG.get("gridmask_ratio", 0.5),
#           )
#       image = preprocess_fn(image)
#       label = tf.one_hot(label, num_classes)
#       return image, label
#
# ═══════════════════════════════════════════════════════════════════════════════


# ── CONFIG additions ─────────────────────────────────────────────────────────────
# CONFIG["use_gridmask"]        = True
# CONFIG["gridmask_probability"] = 0.5    # how often GridMask applies per image
# CONFIG["gridmask_d_min"]       = 40     # grid unit size range in pixels —
# CONFIG["gridmask_d_max"]       = 80     # scale these relative to your img_size
#                                          # (defaults tuned for ~384px images;
#                                          # roughly double for 484px if testing
#                                          # NoisyB0-based specialists like Potato)
# CONFIG["gridmask_ratio"]       = 0.5    # fraction of each grid cell masked

# ═══════════════════════════════════════════════════════════════════════════════
# TRACK 3, TECHNIQUE 3: Copy-Paste (small-patch, hard-label)
# Cuts a small patch from a diseased image and pastes it onto a healthy image
# from the same specialist, keeping the diseased (hard) label unchanged.
# Primary target: Bell Pepper (early-stage, single-lesion misclassification).
#
# IMPORTANT: This requires access to a pool of "healthy"-labeled images to
# paste onto, and only applies when the CURRENT image being augmented is from
# a diseased class. This makes it batch-level (needs multiple images at once)
# rather than a simple single-image map function like the others.
# ═══════════════════════════════════════════════════════════════════════════════

def _copy_paste_numpy(diseased_img, healthy_img, patch_ratio=0.15):
    """
    diseased_img, healthy_img: numpy arrays, same shape (H, W, C)
    patch_ratio: size of the pasted patch as a fraction of image dimensions
                 (0.10-0.20 recommended — deliberately small, to approximate
                 an early-stage/emerging lesion rather than a full late-stage
                 patch. Larger values start to resemble CutMix.)
    Returns: composited image (healthy base + small diseased patch pasted on)
    """
    h, w = diseased_img.shape[0], diseased_img.shape[1]
    patch_h = int(h * patch_ratio)
    patch_w = int(w * patch_ratio)

    # Random source location in the diseased image (where the "lesion-like"
    # patch is cut from — without segmentation, this is a random crop, not
    # a verified lesion location; see documentation note on this limitation)
    src_y = np.random.randint(0, h - patch_h)
    src_x = np.random.randint(0, w - patch_w)
    patch = diseased_img[src_y:src_y + patch_h, src_x:src_x + patch_w].copy()

    # Random destination location on the healthy image
    dst_y = np.random.randint(0, h - patch_h)
    dst_x = np.random.randint(0, w - patch_w)

    result = healthy_img.copy()
    result[dst_y:dst_y + patch_h, dst_x:dst_x + patch_w] = patch
    return result


def copy_paste_augment_batch(images, labels, healthy_class_idx, probability=0.3, patch_ratio=0.15):
    """
    Batch-level copy-paste augmentation. For each image in the batch whose
    label is NOT the healthy class, with probability `probability`, paste a
    small patch from it onto a randomly selected healthy image FROM THE SAME
    BATCH. The augmented image's label remains the ORIGINAL diseased label
    (hard label, unchanged) — this teaches "small lesion on healthy leaf" =
    that disease, not a blended/soft label.

    images: tf.Tensor, shape (B, H, W, C), values in [0, 255] range
    labels: tf.Tensor, shape (B, num_classes), one-hot encoded
    healthy_class_idx: int, the index of the "healthy" class for this specialist
    probability: chance of applying copy-paste to any given eligible image
    patch_ratio: size of the pasted patch as a fraction of image dimensions

    Returns: (augmented_images, labels) — labels are UNCHANGED (hard labels)
    """
    def _apply_batch(images_np, labels_np):
        images_np = images_np.astype(np.uint8)
        batch_size = images_np.shape[0]
        label_indices = np.argmax(labels_np, axis=1)

        healthy_indices = np.where(label_indices == healthy_class_idx)[0]
        if len(healthy_indices) == 0:
            # No healthy images in this batch to paste onto — skip augmentation
            return images_np

        result = images_np.copy()
        for i in range(batch_size):
            if label_indices[i] == healthy_class_idx:
                continue  # only augment DISEASED images (paste happens onto a healthy base)
            if np.random.uniform() >= probability:
                continue

            healthy_idx = np.random.choice(healthy_indices)
            # Diseased image provides the patch source; healthy image is the paste target;
            # result at position i becomes: healthy base + diseased patch, labeled as diseased
            composited = _copy_paste_numpy(images_np[i], images_np[healthy_idx], patch_ratio=patch_ratio)
            result[i] = composited

        return result

    augmented = tf.py_function(
        func=lambda imgs, lbls: _apply_batch(imgs.numpy(), lbls.numpy()),
        inp=[tf.cast(images, tf.uint8), labels],
        Tout=tf.uint8,
    )
    augmented.set_shape(images.shape)
    augmented = tf.cast(augmented, images.dtype)

    return augmented, labels  # labels intentionally unchanged — hard label


# ═══════════════════════════════════════════════════════════════════════════════
# Integration — this is BATCH-LEVEL, so it goes AFTER .batch() in your pipeline,
# similar to how CutMix/MixUp are applied in your existing code, NOT inside
# process_train (which operates on individual unbatched images).
# ═══════════════════════════════════════════════════════════════════════════════
#
# In _apply_preprocessing, after ds.batch(batch_size):
#
#   ds = ds.batch(batch_size)
#   if augment and CONFIG.get("use_copy_paste"):
#       healthy_idx = CONFIG["healthy_class_index"]  # set per specialist, see below
#       ds = ds.map(
#           lambda x, y: copy_paste_augment_batch(
#               x, y, healthy_class_idx=healthy_idx,
#               probability=CONFIG.get("copy_paste_probability", 0.3),
#               patch_ratio=CONFIG.get("copy_paste_patch_ratio", 0.15),
#           ),
#           num_parallel_calls=tf.data.AUTOTUNE,
#       )
#   ds = ds.prefetch(tf.data.AUTOTUNE)
#
# ═══════════════════════════════════════════════════════════════════════════════


# ── CONFIG additions ─────────────────────────────────────────────────────────────
# CONFIG["use_copy_paste"]         = True
# CONFIG["copy_paste_probability"] = 0.3     # chance applied per eligible diseased image
# CONFIG["copy_paste_patch_ratio"] = 0.15    # small patch size, ~10-20% of image dims
# CONFIG["healthy_class_index"]    = 0       # MUST set per specialist — the index of
#                                             # the "healthy" class in that specialist's
#                                             # class_names list (check via class_names.index("...")
#                                             # or print class_names to find it manually)
#
# NOTE: For Bell Pepper specifically (2 classes: "Bell_pepper leaf" = healthy,
# "Bell_pepper leaf spot" = diseased), confirm which index corresponds to
# "Bell_pepper leaf" after your dataloader sorts class names alphabetically —
# print class_names once before setting healthy_class_index to be certain.


def _apply_preprocessing(ds, model_name: str, batch_size: int, augment: bool = False):
    """
    Map preprocessing (and optionally augmentation) over an unbatched dataset,
    then batch and prefetch.
    augment=True  → training set only
    augment=False → validation set (always clean)
    """
    preprocess_fn = _make_preprocess_fn(model_name)
    num_classes   = CONFIG["num_classes"]
    use_randaugment = CONFIG.get("use_randaugment", False)

    if use_randaugment:
        aug_layer = build_randaugment_layer(
            value_range=(0, 255),
            augmentations_per_image=CONFIG.get("randaugment_num_ops", 2),
            magnitude=CONFIG.get("randaugment_magnitude", 0.3),
        ) if CONFIG["use_augmentation"] else None
    else:
        aug_layer = tf.keras.Sequential([
            tf.keras.layers.RandomFlip("horizontal"),
            tf.keras.layers.RandomRotation(0.15),
            tf.keras.layers.RandomZoom(0.15),
            tf.keras.layers.RandomTranslation(0.1, 0.1),
            tf.keras.layers.RandomBrightness(0.2),
            tf.keras.layers.RandomContrast(0.2),
        ], name="augmentation") if CONFIG["use_augmentation"] else None
        
    def process_train(image, label):
        if aug_layer is not None:
            image = aug_layer(image, training=True)
        if CONFIG.get("use_random_filter_augment"):
            image = random_filter_augment(
                image, probability=CONFIG.get("random_filter_probability", 0.3)
            )
        if CONFIG.get("use_gridmask"):
            image = gridmask_augment(
                image,
                probability=CONFIG.get("gridmask_probability", 0.5),
                d_min=CONFIG.get("gridmask_d_min", 40),
                d_max=CONFIG.get("gridmask_d_max", 80),
                ratio=CONFIG.get("gridmask_ratio", 0.5),
            )
        image = preprocess_fn(image)
        label = tf.one_hot(label, num_classes)
        return image, label

    def process_val(image, label):
        image = preprocess_fn(image)
        label = tf.one_hot(label, num_classes)
        return image, label

    fn = process_train if augment else process_val
    ds = ds.map(fn, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)

    #   ds = ds.batch(batch_size)
    if augment and CONFIG.get("use_copy_paste"):
        healthy_idx = CONFIG["healthy_class_index"]  # set per specialist, see below
        ds = ds.map(
            lambda x, y: copy_paste_augment_batch(
                x, y, healthy_class_idx=healthy_idx,
                probability=CONFIG.get("copy_paste_probability", 0.3),
                patch_ratio=CONFIG.get("copy_paste_patch_ratio", 0.15),
            ),
            num_parallel_calls=tf.data.AUTOTUNE,
        )
    ds = ds.prefetch(tf.data.AUTOTUNE)
        
    return ds


# ── CutMix ────────────────────────────────────────────────────────────────────
def _cutmix_batch(images, labels, alpha: float = 1.0):
    batch_size = tf.shape(images)[0]
    lam = np.random.beta(alpha, alpha)
    img_h, img_w = tf.shape(images)[1], tf.shape(images)[2]
    cut_rat = tf.cast(tf.math.sqrt(1.0 - lam), tf.float32)
    cut_h   = tf.cast(tf.cast(img_h, tf.float32) * cut_rat, tf.int32)
    cut_w   = tf.cast(tf.cast(img_w, tf.float32) * cut_rat, tf.int32)
    cx = tf.random.uniform([], 0, img_w, dtype=tf.int32)
    cy = tf.random.uniform([], 0, img_h, dtype=tf.int32)
    x1 = tf.clip_by_value(cx - cut_w // 2, 0, img_w)
    y1 = tf.clip_by_value(cy - cut_h // 2, 0, img_h)
    x2 = tf.clip_by_value(cx + cut_w // 2, 0, img_w)
    y2 = tf.clip_by_value(cy + cut_h // 2, 0, img_h)
    indices = tf.random.shuffle(tf.range(batch_size))
    shuffled_images = tf.gather(images, indices)
    shuffled_labels = tf.gather(labels, indices)
    mask = tf.pad(
        tf.ones([y2 - y1, x2 - x1, tf.shape(images)[3]], dtype=images.dtype),
        [[y1, img_h - y2], [x1, img_w - x2], [0, 0]]
    )
    mixed_images = images * (1 - mask) + shuffled_images * mask
    lam_actual = 1.0 - tf.cast((x2-x1)*(y2-y1), tf.float32) / tf.cast(img_h*img_w, tf.float32)
    mixed_labels = lam_actual * labels + (1.0 - lam_actual) * shuffled_labels
    return mixed_images, mixed_labels


# ── Mixup ─────────────────────────────────────────────────────────────────────
def _mixup_batch(images, labels, alpha: float = 0.2):
    batch_size = tf.shape(images)[0]
    lam = np.random.beta(alpha, alpha)
    indices = tf.random.shuffle(tf.range(batch_size))
    mixed_images = lam * images + (1.0 - lam) * tf.gather(images, indices)
    mixed_labels = lam * labels + (1.0 - lam) * tf.gather(labels, indices)
    return mixed_images, mixed_labels


def apply_cutmix_mixup(ds):
    if CONFIG["use_cutmix"]:
        ds = ds.map(lambda x, y: _cutmix_batch(x, y, CONFIG["cutmix_alpha"]),
                    num_parallel_calls=tf.data.AUTOTUNE)
    if CONFIG["use_mixup"]:
        ds = ds.map(lambda x, y: _mixup_batch(x, y, CONFIG["mixup_alpha"]),
                    num_parallel_calls=tf.data.AUTOTUNE)
    return ds


def get_datasets(model_name: str):
    """
    Main data loading entry point.
    Returns (train_ds, val_ds, class_names)
    """
    data_dir = _get_dataset_directory()
    train_raw, val_raw, class_names = _build_raw_datasets(data_dir)

    train_ds = _apply_preprocessing(train_raw, model_name, CONFIG["batch_size"], augment=True)
    val_ds   = _apply_preprocessing(val_raw,   model_name, CONFIG["batch_size"], augment=False)
    train_ds = apply_cutmix_mixup(train_ds)

    # ── Label sanity check ────────────────────────────────────────────────────
    verify_labels(val_raw.batch(CONFIG["batch_size"]), class_names)

    return train_ds, val_ds, class_names

print("Preprocessing helpers defined.")

## 5. Models

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, Model

# ── MobileViT building blocks ──────────────────────────────────────────────────

def conv_block(x, filters, kernel_size=3, strides=1):
    x = layers.Conv2D(filters, kernel_size, strides=strides, padding="same", use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("swish")(x)
    return x


def inverted_residual_block(x, expanded_channels, output_channels, strides=1):
    m = layers.Conv2D(expanded_channels, 1, padding="same", use_bias=False)(x)
    m = layers.BatchNormalization()(m)
    m = layers.Activation("swish")(m)

    m = layers.DepthwiseConv2D(3, strides=strides, padding="same", use_bias=False)(m)
    m = layers.BatchNormalization()(m)
    m = layers.Activation("swish")(m)

    m = layers.Conv2D(output_channels, 1, padding="same", use_bias=False)(m)
    m = layers.BatchNormalization()(m)

    if strides == 1 and x.shape[-1] == output_channels:
        m = layers.Add()([m, x])
    return m


def transformer_block(x, projection_dim, num_heads=4, mlp_dim=None, dropout=0.1):
    mlp_dim = mlp_dim or projection_dim * 2

    x1 = layers.LayerNormalization(epsilon=1e-6)(x)
    attn = layers.MultiHeadAttention(num_heads=num_heads, key_dim=projection_dim // num_heads, dropout=dropout)(x1, x1)
    x2 = layers.Add()([attn, x])

    x3 = layers.LayerNormalization(epsilon=1e-6)(x2)
    mlp = layers.Dense(mlp_dim, activation="swish")(x3)
    mlp = layers.Dropout(dropout)(mlp)
    mlp = layers.Dense(projection_dim)(mlp)
    mlp = layers.Dropout(dropout)(mlp)
    out = layers.Add()([mlp, x2])
    return out


def mobilevit_block(x, num_blocks, projection_dim, strides=1):
    """
    MobileViT block: local conv representation + global transformer representation, fused.
    """
    local_features = conv_block(x, projection_dim, kernel_size=3, strides=strides)
    local_features = layers.Conv2D(projection_dim, 1, padding="same")(local_features)

    h, w = local_features.shape[1], local_features.shape[2]
    
    # Unfold to patches (flatten spatial dims for transformer)
    non_local = layers.Reshape((h * w, projection_dim))(local_features)
    
    for _ in range(num_blocks):
        non_local = transformer_block(non_local, projection_dim)
    
    non_local = layers.Reshape((h, w, projection_dim))(non_local)
    
    # Fold back and fuse with input
    fused = layers.Concatenate()([x if x.shape[-1] == projection_dim else
                                   layers.Conv2D(projection_dim, 1, padding="same")(x),
                                   non_local])
    fused = layers.Conv2D(projection_dim, 1, padding="same")(fused)
    return fused


def build_mobilevit(num_classes: int, img_size: int = 256) -> Model:
    """
    MobileViT-L — largest variant. Likely to overfit severely on small datasets.
    """
    inputs = tf.keras.Input(shape=(img_size, img_size, 3))

    x = conv_block(inputs, 16, strides=2)

    x = inverted_residual_block(x, 64, 32, strides=1)
    x = inverted_residual_block(x, 128, 64, strides=2)
    x = inverted_residual_block(x, 256, 64, strides=1)

    x = inverted_residual_block(x, 256, 128, strides=2)
    x = mobilevit_block(x, num_blocks=4, projection_dim=192)

    x = inverted_residual_block(x, 512, 256, strides=2)
    x = mobilevit_block(x, num_blocks=8, projection_dim=256)

    x = inverted_residual_block(x, 1024, 320, strides=2)
    x = mobilevit_block(x, num_blocks=4, projection_dim=320)

    x = conv_block(x, 1280, kernel_size=1)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.4)(x)
    outputs = layers.Dense(num_classes, activation="softmax")(x)

    return Model(inputs, outputs, name="MobileViT_L")


In [ ]:
def build_mobilenetv2(num_classes: int) -> Model:
    cfg      = CONFIG["mobilenet_head"]
    img_size = CONFIG["img_size"]
    base     = MobileNetV2(input_shape=(img_size, img_size, 3),
                           include_top=False, weights=MODEL_PATHS["mobilenetv2_imagenet_weights"])
    base.trainable = False

    inputs  = tf.keras.Input(shape=(img_size, img_size, 3))
    x       = base(inputs, training=False)
    x       = layers.GlobalAveragePooling2D()(x)
    x       = layers.Dense(cfg["dense_units"], activation="relu")(x)
    x       = layers.Dropout(cfg["dropout"])(x)
    outputs = layers.Dense(num_classes, activation="softmax")(x)
    return Model(inputs, outputs, name="MobileNetV2_PlantDoc")


def build_efficientnetb0(num_classes: int) -> Model:
    cfg      = CONFIG["efficientnet_head"]
    img_size = CONFIG["img_size"]
    base     = EfficientNetB0(input_shape=(img_size, img_size, 3),
                              include_top=False, weights=MODEL_PATHS["efficientnetb0_imagenet_weights"])
    base.trainable = False

    inputs  = tf.keras.Input(shape=(img_size, img_size, 3))
    x       = base(inputs, training=False)
    x       = layers.GlobalAveragePooling2D()(x)
    x       = layers.BatchNormalization()(x)
    x       = layers.Dense(cfg["dense_units"], activation="relu")(x)
    x       = layers.Dropout(cfg["dropout"])(x)
    outputs = layers.Dense(num_classes, activation="softmax")(x)
    return Model(inputs, outputs, name="EfficientNetB0_PlantDoc")

def build_efficientnetb3(num_classes: int) -> Model:
    cfg      = CONFIG["efficientnet_head"]
    img_size = CONFIG["img_size"]
    base     = EfficientNetB3(
        input_shape=(img_size, img_size, 3),
        include_top=False,
        weights=MODEL_PATHS["efficientnetb3_imagenet_weights"],
    )
    base.trainable = False

    inputs  = tf.keras.Input(shape=(img_size, img_size, 3))
    x       = base(inputs, training=False)
    x       = layers.GlobalAveragePooling2D()(x)
    x       = layers.BatchNormalization()(x)
    x       = layers.Dense(cfg["dense_units"], activation="relu")(x)
    x       = layers.Dropout(cfg["dropout"])(x)
    outputs = layers.Dense(num_classes, activation="softmax")(x)
    return Model(inputs, outputs, name="EfficientNetB3_PlantDoc")

def build_efficientnetv2s(num_classes: int) -> Model:
    cfg      = CONFIG["efficientnet_head"]
    img_size = CONFIG["img_size"]
    base     = EfficientNetV2S(
        input_shape=(img_size, img_size, 3),
        include_top=False,
        weights=MODEL_PATHS["efficientnetv2s_imagenet_weights"],
    )
    base.trainable = False

    inputs  = tf.keras.Input(shape=(img_size, img_size, 3))
    x       = base(inputs, training=False)
    x       = layers.GlobalAveragePooling2D()(x)
    x       = layers.BatchNormalization()(x)
    x       = layers.Dense(cfg["dense_units"], activation="relu")(x)
    x       = layers.Dropout(cfg["dropout"])(x)
    outputs = layers.Dense(num_classes, activation="softmax")(x)
    return Model(inputs, outputs, name="EfficientNetV2S_PlantDoc")



def build_inceptionresnetv2(num_classes: int) -> Model:
    img_size = CONFIG["img_size"]
    base = InceptionResNetV2(
        input_shape=(img_size, img_size, 3),
        include_top=False,
        weights=MODEL_PATHS["inceptionresnetv2_imagenet_weights"],
    )
    base.trainable = False

    inputs  = tf.keras.Input(shape=(img_size, img_size, 3))
    x       = base(inputs, training=False)
    x       = layers.GlobalAveragePooling2D()(x)
    x       = layers.BatchNormalization()(x)
    x       = layers.Dense(512, activation="relu")(x)
    x       = layers.Dropout(0.5)(x)
    outputs = layers.Dense(num_classes, activation="softmax")(x)
    return Model(inputs, outputs, name="InceptionResNetV2_PlantDoc")

def build_convnextbase(num_classes: int) -> Model:
    img_size = CONFIG["img_size"]
    base = ConvNeXtBase(
        input_shape=(img_size, img_size, 3),
        include_top=False,
        weights=MODEL_PATHS["convnextbase_imagenet_weights"],
    )
    base.trainable = False

    inputs  = tf.keras.Input(shape=(img_size, img_size, 3))
    x       = base(inputs, training=False)
    x       = layers.GlobalAveragePooling2D()(x)
    x       = layers.LayerNormalization()(x)   # ConvNeXt uses LayerNorm not BatchNorm
    x       = layers.Dense(512, activation="relu")(x)
    x       = layers.Dropout(0.5)(x)
    outputs = layers.Dense(num_classes, activation="softmax")(x)
    return Model(inputs, outputs, name="ConvNeXtBase_PlantDoc")

def build_efficientnetv2l(num_classes: int) -> Model:
    img_size = CONFIG["img_size"]
    base = EfficientNetV2L(
        input_shape=(img_size, img_size, 3),
        include_top=False,
        weights=MODEL_PATHS["efficientnetv2l_imagenet_weights"],
    )
    base.trainable = False

    inputs  = tf.keras.Input(shape=(img_size, img_size, 3))
    x       = base(inputs, training=False)
    x       = layers.GlobalAveragePooling2D()(x)
    x       = layers.BatchNormalization()(x)
    x       = layers.Dense(512, activation="relu")(x)
    x       = layers.Dropout(0.5)(x)
    outputs = layers.Dense(num_classes, activation="softmax")(x)
    return Model(inputs, outputs, name="EfficientNetV2L_PlantDoc")

def build_convnextlarge(num_classes: int) -> Model:
    img_size = CONFIG["img_size"]
    base = ConvNeXtLarge(
        input_shape=(img_size, img_size, 3),
        include_top=False,
        weights=MODEL_PATHS["convnextlarge_imagenet_weights"],
    )
    base.trainable = False

    inputs  = tf.keras.Input(shape=(img_size, img_size, 3))
    x       = base(inputs, training=False)
    x       = layers.GlobalAveragePooling2D()(x)
    x       = layers.LayerNormalization()(x)
    x       = layers.Dense(512, activation="relu")(x)
    x       = layers.Dropout(0.5)(x)
    outputs = layers.Dense(num_classes, activation="softmax")(x)
    return Model(inputs, outputs, name="ConvNeXtLarge_PlantDoc")

def build_nasnetlarge(num_classes: int) -> Model:
    img_size = CONFIG["img_size"]
    base = NASNetLarge(
        input_shape=(img_size, img_size, 3),
        include_top=False,
        weights=MODEL_PATHS["nasnetlarge_imagenet_weights"],
    )
    base.trainable = False

    inputs  = tf.keras.Input(shape=(img_size, img_size, 3))
    x       = base(inputs, training=False)
    x       = layers.GlobalAveragePooling2D()(x)
    x       = layers.BatchNormalization()(x)
    x       = layers.Dense(512, activation="relu")(x)
    x       = layers.Dropout(0.5)(x)
    outputs = layers.Dense(num_classes, activation="softmax")(x)
    return Model(inputs, outputs, name="NASNetLarge_PlantDoc")

def build_noisy_model(num_classes: int) -> Model:
    img_size = CONFIG["img_size"]
    
    base = EfficientNetB0(
        input_shape=(img_size, img_size, 3),
        include_top=False,
        weights=None,
    )
    base.load_weights(
        CONFIG["noisy_student_weights"],
        by_name=True,
        skip_mismatch=True
    )
    base.trainable = False

    inputs  = tf.keras.Input(shape=(img_size, img_size, 3))
    x       = base(inputs, training=False)
    x       = layers.GlobalAveragePooling2D()(x)
    x       = layers.BatchNormalization()(x)
    x       = layers.Dropout(0.3)(x)
    x       = layers.Dense(128, activation="relu")(x)
    x       = layers.Dropout(0.2)(x)
    x       = layers.Dense(num_classes)(x)
    outputs = layers.Activation("softmax", dtype="float32")(x)
    return Model(inputs, outputs, name="NoisyStudent_PlantDoc")

def build_tomato_from_rest_weights(num_classes: int) -> Model:
    rest_model_path = CONFIG["rest_model_path"]
    print(f"[Transfer] Loading Rest model: {rest_model_path}")
    rest_model = tf.keras.models.load_model(rest_model_path)

    base_layer = None
    for layer in rest_model.layers:
        if hasattr(layer, 'layers'):
            base_layer = layer
            break

    if base_layer is None:
        raise ValueError("Could not find backbone layer in Rest model.")

    print(f"[Transfer] Extracted backbone: {base_layer.name} ({len(base_layer.layers)} layers)")
    base_layer.trainable = CONFIG.get("fine_tune", False)

    img_size = CONFIG["img_size"]
    inputs  = tf.keras.Input(shape=(img_size, img_size, 3))
    x       = base_layer(inputs, training=False)
    x       = layers.GlobalAveragePooling2D()(x)
    x       = layers.LayerNormalization()(x)
    x       = layers.Dense(512, activation="relu")(x)
    x       = layers.Dropout(0.5)(x)
    outputs = layers.Dense(num_classes, activation="softmax")(x)

    return Model(inputs, outputs, name="Tomato_from_Rest_transfer")

def build_convnexttiny(num_classes: int) -> Model:
    img_size = CONFIG["img_size"]
    base = ConvNeXtTiny(
        input_shape=(img_size, img_size, 3),
        include_top=False,
        weights=MODEL_PATHS["convnexttiny_imagenet_weights"],
    )
    base.trainable = False

    inputs  = tf.keras.Input(shape=(img_size, img_size, 3))
    x       = base(inputs, training=False)
    x       = layers.GlobalAveragePooling2D()(x)
    x       = layers.LayerNormalization()(x)
    x       = layers.Dense(512, activation="relu")(x)
    x       = layers.Dropout(0.5)(x)
    outputs = layers.Dense(num_classes, activation="softmax")(x)
    return Model(inputs, outputs, name="ConvNeXtTiny_PlantDoc")


def build_convnextsmall(num_classes: int) -> Model:
    img_size = CONFIG["img_size"]
    base = ConvNeXtSmall(
        input_shape=(img_size, img_size, 3),
        include_top=False,
        weights=MODEL_PATHS["convnextsmall_imagenet_weights"],
    )
    base.trainable = False

    inputs  = tf.keras.Input(shape=(img_size, img_size, 3))
    x       = base(inputs, training=False)
    x       = layers.GlobalAveragePooling2D()(x)
    x       = layers.LayerNormalization()(x)
    x       = layers.Dense(512, activation="relu")(x)
    x       = layers.Dropout(0.5)(x)
    outputs = layers.Dense(num_classes, activation="softmax")(x)
    return Model(inputs, outputs, name="ConvNeXtSmall_PlantDoc")

def build_from_plantvillage_pretrained(plantvillage_model_path, num_classes: int) -> Model:
    pv_model = tf.keras.models.load_model(plantvillage_model_path)
 
    # Extract the backbone (everything except the final classification layer)
    base_layer = None
    for layer in pv_model.layers:
        if hasattr(layer, 'layers'):  # nested backbone (e.g. ConvNeXtBase)
            base_layer = layer
            break
 
    if base_layer is None:
        raise ValueError(
            f"Could not find a nested backbone layer in {plantvillage_model_path}. "
            f"This model may use a flatter architecture (e.g. NoisyB0 built without a "
            f"nested sub-model) — inspect pv_model.layers manually and adjust the "
            f"extraction logic accordingly."
        )
 
    print(f"[PV Transfer] Loaded PlantVillage model: {plantvillage_model_path}")
    print(f"[PV Transfer] Extracted backbone: {base_layer.name} ({len(base_layer.layers)} layers)")
 
    base_layer.trainable = CONFIG.get("fine_tune", False)  # frozen by default, consistent with established pattern
 
    img_size = CONFIG["img_size"]
    inputs  = tf.keras.Input(shape=(img_size, img_size, 3))
    x       = base_layer(inputs, training=False)
    x       = layers.GlobalAveragePooling2D()(x)
    x       = layers.LayerNormalization()(x)
    x       = layers.Dense(512, activation="relu")(x)
    x       = layers.Dropout(0.5)(x)
    outputs = layers.Dense(num_classes, activation="softmax")(x)  # NEW head, PlantDoc's class count
 
    return Model(inputs, outputs, name="PlantVillage_to_PlantDoc_transfer")

def get_model(model_name: str, num_classes: int = None) -> Model:
    if num_classes is None:
        num_classes = CONFIG["num_classes"]
    if model_name == "mobilenetv2":
        return build_mobilenetv2(num_classes)
    elif model_name == "efficientnetb0":
        return build_efficientnetb0(num_classes)
    elif model_name == "efficientnetb3":
        return build_efficientnetb3(num_classes)
    elif model_name == "EfficientNetV2S":
        return build_efficientnetv2s(num_classes)
    elif model_name == "inceptionresnetv2":
        return build_inceptionresnetv2(num_classes)
    elif model_name == "efficientnetv2l":
        return build_efficientnetv2l(num_classes)
    elif model_name == "convnextbase":
        return build_convnextbase(num_classes)
    elif model_name == "ConvNeXtLarge":
        return build_convnextlarge(num_classes)
    elif model_name == "NASNetLarge":
        return build_nasnetlarge(num_classes)
    elif model_name == "noisy":
        return build_noisy_model(num_classes)
    elif model_name == "clipconvnextbase":
        return build_convnextbase_clip(num_classes)
    elif model_name == "DenseNet201":
        return build_densenet201(num_classes)
    elif model_name == "mobilevit":
        return build_mobilevit(num_classes, img_size=CONFIG["img_size"])
    elif model_name == "tomato_from_rest":
        return build_tomato_from_rest_weights(num_classes)
    elif model_name == "convnexttiny":
        return build_convnexttiny(num_classes)
    elif model_name == "convnextsmall":
        return build_convnextsmall(num_classes)
    elif model_name == "plantvillage_transfer":
        return build_from_plantvillage_pretrained(
            plantvillage_model_path=CONFIG["plantvillage_model_path"],
            num_classes=num_classes
        )
    else:
        raise ValueError(f"Unknown model: {model_name}")


def unfreeze_model(model: Model, model_name: str):
    for layer in model.layers:
        if hasattr(layer, 'layers'):
            layer.trainable = True
            print(f"[FineTune] Unfreezing base: {layer.name} ({len(layer.layers)} layers)")
            break
    return model


def compile_model(model: Model, lr: float, num_classes: int = None, class_counts: list = None):
    if CONFIG.get("use_focal_loss") and class_counts:
        total = sum(class_counts)
        alpha = [total / (len(class_counts) * c) for c in class_counts]
        alpha = [a / sum(alpha) for a in alpha]
        loss_fn = focal_loss(gamma=CONFIG["focal_loss_gamma"], alpha=alpha)
        print(f"[FocalLoss] Using alpha weights: {alpha}")
    else:
        loss_fn = "categorical_crossentropy"

    if CONFIG.get("use_cyclical_lr"):
        lr_schedule = tf.keras.optimizers.schedules.CosineDecayRestarts(
            initial_learning_rate=lr,
            first_decay_steps=CONFIG.get("cyclical_first_decay_steps", 15),
            t_mul=CONFIG.get("cyclical_t_mul", 1.5),
            m_mul=CONFIG.get("cyclical_m_mul", 0.85),
        )
        optimizer = tf.keras.optimizers.Adam(learning_rate=lr_schedule)
        print(f"[CyclicalLR] Using CosineDecayRestarts schedule")
    else:
        optimizer = tf.keras.optimizers.Adam(learning_rate=lr)

    model.compile(
        optimizer=optimizer,
        loss=loss_fn,
        metrics=["accuracy"],
    )
    return model


class FineTuneCallback(tf.keras.callbacks.Callback):
    """Unfreezes base model at fine_tune_at_epoch and recompiles with lower LR."""
    def __init__(self, model_name: str):
        super().__init__()
        self.model_name       = model_name
        self.trigger_epoch    = CONFIG["fine_tune_at_epoch"]
        self.fine_tune_lr     = CONFIG["fine_tune_lr"]
        self.already_unfrozen = False

    def on_epoch_begin(self, epoch, logs=None):
        if CONFIG["fine_tune"] and epoch == self.trigger_epoch and not self.already_unfrozen:
            print(f"\n[FineTune] Epoch {epoch}: Unfreezing for fine-tuning.")
            unfreeze_model(self.model, self.model_name)
            self.model.optimizer.learning_rate = self.fine_tune_lr
            print(f"[FineTune] LR set to {self.fine_tune_lr}")
            self.already_unfrozen = True

print("Model builders defined.")

## 6. Training

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# 6. TRAINING — callbacks and single-run training
#   Checkpoint selection monitors CONFIG["checkpoint_monitor"] (val_loss on the
#   held-out validation set). LR-plateau and early-stopping settings come from CONFIG.
# ═══════════════════════════════════════════════════════════════════════════════
from sklearn.metrics import precision_recall_fscore_support

class LRLoggerCallback(tf.keras.callbacks.Callback):
    def on_epoch_end(self, epoch, logs=None):
        lr = self.model.optimizer.learning_rate
        if callable(lr):  # it's a schedule
            current_lr = lr(self.model.optimizer.iterations)
        else:
            current_lr = lr
        logs = logs or {}
        logs['lr'] = float(current_lr)


def save_config(experiment_name: str = None):
    exp = experiment_name or CONFIG["experiment_name"]
    path = os.path.join(CONFIG["output_dir"], exp, "config.txt")
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w") as f:
        f.write(f"========== Config: {exp} ==========\n")
        for k, v in CONFIG.items():
            f.write(f"  {k}: {v}\n")
        f.write("=====================================\n")
    print(f"[Config] Saved to {path}")


class MetricsSummaryCallback(tf.keras.callbacks.Callback):
    def __init__(self, val_ds, class_names):
        super().__init__()
        self.val_ds      = val_ds
        self.class_names = class_names

    def on_train_end(self, logs=None):
        print("\n========== Final Metrics ==========")
        all_preds, all_labels = [], []
        for images, labels in self.val_ds:
            preds = self.model.predict(images, verbose=0)
            all_preds.append(np.argmax(preds, axis=1))
            all_labels.append(np.argmax(labels.numpy(), axis=1))

        y_pred = np.concatenate(all_preds)
        y_true = np.concatenate(all_labels)

        precision, recall, f1, _ = precision_recall_fscore_support(
            y_true, y_pred, average="weighted", zero_division=0
        )
        acc = np.mean(y_pred == y_true)

        print(f"  Val Accuracy : {acc:.4f}")
        print(f"  Precision    : {precision:.4f}")
        print(f"  Recall       : {recall:.4f}")
        print(f"  F1 Score     : {f1:.4f}")
        print("===================================\n")


def focal_loss(gamma=2.0, alpha=None):
    """
    Focal loss for multi-class classification.
    gamma: focusing parameter — higher values focus more on hard examples
    alpha: per-class weight vector (list), or None for uniform weighting
    """
    def loss_fn(y_true, y_pred):
        y_pred = tf.clip_by_value(y_pred, 1e-7, 1.0 - 1e-7)
        
        if alpha is not None:
            alpha_tensor = tf.constant(alpha, dtype=tf.float32)
            alpha_weight = tf.reduce_sum(y_true * alpha_tensor, axis=-1)
        else:
            alpha_weight = 1.0

        cross_entropy = -y_true * tf.math.log(y_pred)
        weight = tf.pow(1 - y_pred, gamma)
        loss = alpha_weight[:, tf.newaxis] * weight * cross_entropy if alpha is not None else weight * cross_entropy
        loss = tf.reduce_sum(loss, axis=-1)
        return tf.reduce_mean(loss)
    
    return loss_fn


def get_callbacks(model_name: str, fold: int = None, use_cyclical: bool = False, seed_suffix: str = ""):
    """seed_suffix (e.g. "_seed42") keeps save paths / CSV logs of different seeds apart."""
    suffix = f"_fold{fold}" if fold is not None else ""
    suffix = f"{suffix}{seed_suffix}"
    save_path = get_model_save_path(f"{model_name}{suffix}")
    log_path = os.path.join(CONFIG["output_dir"], CONFIG["experiment_name"], f"{model_name}{suffix}_history.csv")
    os.makedirs(os.path.dirname(log_path), exist_ok=True)
    monitor = CONFIG["checkpoint_monitor"]
    mode = "min" if "loss" in monitor else "max"

    cbs = []
    if save_path is not None:
        cbs.append(tf.keras.callbacks.ModelCheckpoint(
            filepath=save_path, monitor=monitor, mode=mode, save_best_only=True, verbose=1))
    else:
        print(f"[get_callbacks] save_model=False — no checkpoint for {model_name}{suffix}")
    cbs.append(tf.keras.callbacks.CSVLogger(log_path, append=False))
    if not use_cyclical:
        cbs.append(tf.keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss", factor=CONFIG["reduce_lr_factor"], patience=CONFIG["reduce_lr_patience"],
            min_lr=CONFIG["reduce_lr_min_lr"], verbose=1))
    if CONFIG["early_stopping"]:
        cbs.append(tf.keras.callbacks.EarlyStopping(
            monitor=monitor, mode=mode, patience=CONFIG["early_stopping_patience"],
            restore_best_weights=True, verbose=1))
    cbs.append(LRLoggerCallback())
    return cbs


def train_single_split(model_name: str, seed_suffix: str = ""):
    save_config()
    print(f"\n{'=' * 50}\n Training: {model_name.upper()} ({CONFIG['component']}){seed_suffix}\n{'=' * 50}")
    train_ds, val_ds, class_names = get_datasets(model_name)
    model = get_model(model_name, len(class_names))
    model = compile_model(model, CONFIG["learning_rate"], class_counts=_train_class_counts())
    model.summary()
    cbs = get_callbacks(model_name, use_cyclical=CONFIG["use_cyclical_lr"], seed_suffix=seed_suffix)
    cbs.append(MetricsSummaryCallback(val_ds, class_names))
    if CONFIG["fine_tune"]:
        cbs.append(FineTuneCallback(model_name))
    history = model.fit(train_ds, validation_data=val_ds, epochs=CONFIG["epochs"],
                        callbacks=cbs, verbose=CONFIG["verbose"])
    monitor = CONFIG["checkpoint_monitor"]
    vals = history.history.get(monitor, [])
    if vals:
        best_epoch = int(np.argmin(vals) if "loss" in monitor else np.argmax(vals)) + 1
        print(f"\n[Result] {model_name}{seed_suffix} — best {monitor} {vals[best_epoch - 1]:.4f} at epoch {best_epoch}")
    return model, history, class_names


print("Training functions defined.")


## 7. Multi-seed runner

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# 7. MULTI-SEED RUNNER
#   Trains once per seed, keeps every checkpoint, reloads the best one and reports
#   validation (val_*) and test (test_*) metrics separately. Progress is saved after
#   every seed, so a crashed or interrupted session resumes where it stopped.
# ═══════════════════════════════════════════════════════════════════════════════
import json
import gc
import time
from scipy import stats
import random as _random_module

def set_all_seeds(seed: int):
    """Set every relevant random seed for a single run."""
    _random_module.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)


def _compute_run_metrics(model, val_ds, class_names):
    """Full evaluation on val_ds, returning a metrics dict for one trained model."""
    from sklearn.metrics import balanced_accuracy_score, f1_score

    all_preds, all_labels = [], []
    for images, labels in val_ds:
        preds = model.predict(images, verbose=0)
        all_preds.append(np.argmax(preds, axis=1))
        all_labels.append(np.argmax(labels.numpy(), axis=1))

    y_pred = np.concatenate(all_preds)
    y_true = np.concatenate(all_labels)

    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average="weighted", zero_division=0
    )
    macro_f1 = f1_score(y_true, y_pred, average="macro", zero_division=0)
    bal_acc  = balanced_accuracy_score(y_true, y_pred)
    acc      = float(np.mean(y_pred == y_true))

    return {
        "val_accuracy": acc,
        "precision": float(precision),
        "recall": float(recall),
        "f1": float(f1),
        "macro_f1": float(macro_f1),
        "balanced_accuracy": float(bal_acc),
    }


def _prefixed(metrics, prefix):
    return {f"{prefix}_{'accuracy' if k == 'val_accuracy' else k}": v for k, v in metrics.items()}


def progress_path(component=None):
    exp = f"{CONFIG_BASE['experiment_prefix']}_{component}" if component else CONFIG["experiment_name"]
    return os.path.join(CONFIG_BASE["output_dir"], exp, "progress.json")


def run_multi_seed_experiment(model_name: str, confidence_level: float = 0.95, resume_path: str = None):
    seeds = CONFIG.get("multi_seed_list")
    if not seeds:
        raise ValueError("CONFIG['multi_seed_list'] is empty.")
    if not CONFIG.get("save_model", True):
        raise ValueError("CONFIG['save_model'] must be True: the best checkpoint is reloaded for evaluation.")
    resume_path = resume_path or progress_path()
    all_results, done = [], set()
    if os.path.exists(resume_path):
        all_results = json.load(open(resume_path)).get("per_seed_results", [])
        done = {r["seed"] for r in all_results}
        print(f"[MultiSeed] Resuming — completed seeds: {sorted(done)}")

    for seed in [s for s in seeds if s not in done]:
        print(f"\n{'=' * 70}\n  MULTI-SEED RUN — {CONFIG['component']} / {model_name}, seed={seed}\n{'=' * 70}")
        CONFIG["random_seed"] = seed
        set_all_seeds(seed)
        ckpt = os.path.join(CONFIG["output_dir"], CONFIG["experiment_name"],
                            f"{model_name}_seed{seed}{CONFIG['save_format']}")
        if os.path.exists(ckpt):     # left by a crashed attempt: keep it aside, retrain from scratch
            aside = f"{ckpt}.incomplete_{time.strftime('%Y%m%d_%H%M%S')}"
            os.rename(ckpt, aside)
            print(f"[MultiSeed] Found a checkpoint from an interrupted attempt; moved to {aside}")
        model, history, class_names = train_single_split(model_name, seed_suffix=f"_seed{seed}")
        if not os.path.exists(ckpt):
            raise RuntimeError(f"Expected checkpoint not found: {ckpt}")
        del model
        tf.keras.backend.clear_session()
        model = tf.keras.models.load_model(ckpt)
        _, val_ds, _ = get_datasets(model_name)
        test_ds, test_names = get_test_dataset(model_name)
        if test_names != class_names:
            raise RuntimeError("Test class order differs from training class order.")
        monitor = CONFIG["checkpoint_monitor"]
        vl = history.history.get(monitor, [])
        best = (int(np.argmin(vl) if "loss" in monitor else np.argmax(vl)) + 1) if vl else None
        result = {"seed": seed, "checkpoint_path": ckpt, "selected_epoch": best,
                  **_prefixed(_compute_run_metrics(model, val_ds, class_names), "val"),
                  **_prefixed(_compute_run_metrics(model, test_ds, class_names), "test")}
        all_results.append(result)
        print(f"[MultiSeed] seed={seed}: val_accuracy={result['val_accuracy']:.4f} | "
              f"test_accuracy={result['test_accuracy']:.4f} | selected epoch {best}")
        del model
        tf.keras.backend.clear_session()
        gc.collect()
        os.makedirs(os.path.dirname(resume_path), exist_ok=True)
        json.dump({"component": CONFIG["component"], "model_name": model_name, "seeds": seeds,
                   "per_seed_results": all_results}, open(resume_path, "w"), indent=2)

    todo = [r for r in all_results if r["seed"] in seeds]
    if len(todo) < len(seeds):
        return {"model_name": model_name, "seeds": seeds, "per_seed_results": todo, "summary": None, "complete": False}
    metric_names = [k for k in todo[0] if k.startswith(("val_", "test_"))]
    summary = {}
    for name in metric_names:
        v = np.array([r[name] for r in todo], dtype=float)
        mean, std = v.mean(), (v.std(ddof=1) if len(v) > 1 else 0.0)
        ci = (stats.t.interval(confidence_level, len(v) - 1, loc=mean, scale=stats.sem(v))
              if len(v) > 1 and std > 0 else (mean, mean))
        summary[name] = {"mean": float(mean), "std": float(std), "ci_low": float(ci[0]),
                         "ci_high": float(ci[1]), "values": v.tolist()}
    print(f"\n{'=' * 70}\n  SUMMARY — {CONFIG['component']} / {model_name}, seeds {seeds}\n{'=' * 70}")
    for name in ("test_accuracy", "test_macro_f1", "val_accuracy"):
        s = summary[name]
        print(f"  {name:16s}: {s['mean']:.4f} ± {s['std']:.4f}  (95% CI [{s['ci_low']:.4f}, {s['ci_high']:.4f}])")
    return {"component": CONFIG["component"], "model_name": model_name, "seeds": seeds,
            "per_seed_results": todo, "summary": summary, "complete": True}


def save_multi_seed_results(results: dict, filename: str = None):
    import json as _json
    fname = filename or f"{results['model_name']}_multiseed_results.json"
    path = os.path.join(CONFIG["output_dir"], CONFIG["experiment_name"], fname)
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w") as f:
        _json.dump(results, f, indent=2)
    print(f"\n[MultiSeed] Results saved to {path}")


print("Multi-seed runner defined.")


## 8. Assembled-pipeline evaluation (definitions)

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# 8. EVALUATE THE ASSEMBLED PIPELINE — per seed, once all 8 components are trained
#   Checkpoints, class names and image sizes all come from COMPONENTS, so there are
#   no paths to type. Images are loaded exactly like training (TF decode + bilinear
#   resize) and preprocessed with the same _make_preprocess_fn as training.
#   Outputs go to <output_dir>/<prefix>_pipeline/.
# ═══════════════════════════════════════════════════════════════════════════════
import pandas as pd

PIPELINE_DIR = os.path.join(CONFIG_BASE["output_dir"], f"{CONFIG_BASE['experiment_prefix']}_pipeline")


def _load_tf_image(path, size):
    img = tf.image.decode_image(tf.io.read_file(path), channels=3, expand_animations=False)
    return tf.image.resize(img, [size, size], method="bilinear").numpy().astype(np.float32)


def component_checkpoint(name, seed):
    p = COMPONENTS[name]
    return os.path.join(CONFIG_BASE["output_dir"], f"{CONFIG_BASE['experiment_prefix']}_{name}",
                        f"{p['model']}_seed{seed}{CONFIG_BASE['save_format']}")


def component_class_names(name):
    p = COMPONENTS[name]
    if p["type"] == "classifier":
        return _class_dirs(build_classifier_dataset_v2(include_rest=p["include_rest"])["train"])
    return _class_dirs(_filtered_split_dirs(p["keywords"], src=final_split_dirs())["train"])


def _predict(name, paths, seed, batch_size=8):
    p = COMPONENTS[name]
    model = tf.keras.models.load_model(component_checkpoint(name, seed))
    names = component_class_names(name)
    if model.output_shape[-1] != len(names):
        raise RuntimeError(f"{name}: model has {model.output_shape[-1]} outputs but {len(names)} classes {names}")
    prep = _make_preprocess_fn(p["model"])
    probs = []
    for i in range(0, len(paths), batch_size):
        batch = np.stack([_load_tf_image(q, p["img_size"]) for q in paths[i:i + batch_size]])
        probs.append(np.asarray(model(prep(batch), training=False)))
    del model
    tf.keras.backend.clear_session()
    gc.collect()
    probs = np.concatenate(probs) if probs else np.zeros((0, len(names)))
    return [names[k] for k in probs.argmax(1)], probs.max(1) if len(probs) else np.zeros(0)


def evaluate_pipeline(seed):
    classifier = next(n for n, p in COMPONENTS.items() if p["type"] == "classifier")
    missing = [n for n in COMPONENTS if not os.path.exists(component_checkpoint(n, seed))]
    if missing:
        raise RuntimeError(f"Seed {seed}: no checkpoint yet for {missing}")
    test_dir = final_split_dirs()["test"]
    rows = [{"image": os.path.join(test_dir, c, f), "true_class": c, "true_species": species_of(c)}
            for c in _class_dirs(test_dir) for f in sorted(os.listdir(os.path.join(test_dir, c)))
            if f.lower().endswith(TF_IMAGE_EXT)]
    df = pd.DataFrame(rows)
    print(f"[Pipeline] seed {seed}: {len(df)} test images")

    df["routed_to"], df["classifier_confidence"] = _predict(classifier, list(df["image"]), seed)
    df["routing_correct"] = df["routed_to"] == df["true_species"]
    df["predicted_class"] = None
    for name in [n for n, p in COMPONENTS.items() if p["type"] != "classifier"]:
        idx = df.index[df["routed_to"] == name]
        if len(idx):
            df.loc[idx, "predicted_class"], _ = _predict(name, list(df.loc[idx, "image"]), seed)
    unknown = set(df["routed_to"]) - set(COMPONENTS)
    if unknown:
        raise RuntimeError(f"Classifier produced labels with no component: {unknown}")
    df["final_correct"] = df["predicted_class"] == df["true_class"]

    n = len(df)
    routing, final = int(df["routing_correct"].sum()), int(df["final_correct"].sum())
    print(f"\nRouting accuracy : {routing}/{n} = {100 * routing / n:.2f}%")
    print(f"Final accuracy   : {final}/{n} = {100 * final / n:.2f}%")
    print("\nRouting confusion matrix (rows = true species, cols = routed to):")
    print(pd.crosstab(df["true_species"], df["routed_to"]).to_string())
    per_species = (df.groupby("true_species")
                     .agg(images=("image", "size"), routed_ok=("routing_correct", "sum"), correct=("final_correct", "sum")))
    ok = df[df["routing_correct"]].groupby("true_species")["final_correct"].agg(["sum", "size"])
    per_species["specialist_acc_on_routed"] = ok["sum"] / ok["size"]
    per_species["end_to_end_acc"] = per_species["correct"] / per_species["images"]
    print("\nPer species:\n" + per_species.round(4).to_string())

    os.makedirs(PIPELINE_DIR, exist_ok=True)
    df.to_csv(os.path.join(PIPELINE_DIR, f"pipeline_log_seed{seed}.csv"), index=False)
    summary = {"seed": seed, "n": n, "routing_correct": routing, "final_correct": final,
               "routing_accuracy": routing / n, "final_accuracy": final / n}
    json.dump(summary, open(os.path.join(PIPELINE_DIR, f"pipeline_summary_seed{seed}.json"), "w"), indent=2)
    return df, summary


def theoretical_pipeline(seed, pipeline_df=None):
    """Theoretical accuracy from each component's own test accuracy for this seed."""
    acc = {}
    for name in COMPONENTS:
        path = progress_path(name)
        rows = [r for r in json.load(open(path))["per_seed_results"] if r["seed"] == seed] if os.path.exists(path) else []
        if not rows:
            raise RuntimeError(f"No seed-{seed} result for {name} ({path})")
        acc[name] = rows[0]["test_accuracy"]
    test_dir = final_split_dirs()["test"]
    N = {}
    for c in _class_dirs(test_dir):
        N[species_of(c)] = N.get(species_of(c), 0) + sum(f.lower().endswith(TF_IMAGE_EXT)
                                                         for f in os.listdir(os.path.join(test_dir, c)))
    total = sum(N.values())
    classifier = next(n for n, p in COMPONENTS.items() if p["type"] == "classifier")
    perfect = sum(N[s] * acc[s] for s in N) / total
    out = {"seed": seed, "perfect_routing": perfect, "simple": acc[classifier] * perfect}
    if pipeline_df is not None:
        recall = pipeline_df.groupby("true_species")["routing_correct"].mean()
        out["class_aware"] = sum(N[s] * recall[s] * acc[s] for s in N) / total
    print(f"[Theory] seed {seed}: perfect routing {100 * out['perfect_routing']:.2f}% | "
          f"simple (c7 x) {100 * out['simple']:.2f}%"
          + (f" | class-aware {100 * out['class_aware']:.2f}%" if "class_aware" in out else ""))
    json.dump(out, open(os.path.join(PIPELINE_DIR, f"theoretical_seed{seed}.json"), "w"), indent=2)
    return out


def summarize_pipeline(seeds, confidence_level=0.95):
    """Mean ± std and t-interval over seeds, measured and theoretical."""
    rows = []
    for s in seeds:
        m = json.load(open(os.path.join(PIPELINE_DIR, f"pipeline_summary_seed{s}.json")))
        t = json.load(open(os.path.join(PIPELINE_DIR, f"theoretical_seed{s}.json")))
        rows.append({"seed": s, "measured": m["final_accuracy"], "routing": m["routing_accuracy"],
                     "theory_simple": t["simple"], "theory_class_aware": t.get("class_aware", np.nan)})
    df = pd.DataFrame(rows)
    print(df.round(4).to_string(index=False))
    for col in ("measured", "routing", "theory_simple", "theory_class_aware"):
        v = df[col].dropna().to_numpy()
        if len(v) > 1 and v.std(ddof=1) > 0:
            lo, hi = stats.t.interval(confidence_level, len(v) - 1, loc=v.mean(), scale=stats.sem(v))
            print(f"  {col:18s}: {100 * v.mean():.2f}% ± {100 * v.std(ddof=1):.2f}%  (95% CI [{100 * lo:.2f}%, {100 * hi:.2f}%])")
        elif len(v):
            print(f"  {col:18s}: {100 * v.mean():.2f}% (n={len(v)})")
    return df


print("Pipeline evaluation defined.")


## 9. Run everything
**9a** defines the orchestrator; **9b** runs every component for every seed (Classifier first, seed by seed), evaluates each finished seed, and summarizes. Safe to re-run: finished runs are skipped.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# 9a. RUN-ALL ORCHESTRATOR (definitions)
#   Seed by seed: every component for seed 42 (Classifier first), then the seed-42
#   pipeline evaluation, then the next seed. Memory is cleared around every run.
#   A failed run is logged and skipped; re-running retries only what is missing.
#   State: <output_dir>/<prefix>_run_all_state.json (plus each component's progress.json).
#   If a component's preset changed, its old results (and all pipeline evaluations) are moved to
#   <prefix>_superseded/ automatically, and that component is retrained for every seed.
# ═══════════════════════════════════════════════════════════════════════════════
import time
import traceback
from datetime import datetime

RUN_ALL_STATE = os.path.join(CONFIG_BASE["output_dir"], f"{CONFIG_BASE['experiment_prefix']}_run_all_state.json")


def free_memory():
    tf.keras.backend.clear_session()
    gc.collect()


def seed_done(component, seed):
    p = progress_path(component)
    return os.path.exists(p) and any(r["seed"] == seed for r in json.load(open(p))["per_seed_results"])


def pipeline_done(seed):
    return all(os.path.exists(os.path.join(PIPELINE_DIR, f)) for f in
               (f"pipeline_summary_seed{seed}.json", f"theoretical_seed{seed}.json"))


def _save_state(state):
    os.makedirs(os.path.dirname(RUN_ALL_STATE), exist_ok=True)
    json.dump(state, open(RUN_ALL_STATE, "w"), indent=2)



# ── Stale results: saved results produced with settings that differ from the current preset ──
STALE_KEYS = ("models_to_train", "img_size", "epochs", "learning_rate", "fine_tune", "fine_tune_lr",
              "fine_tune_at_epoch", "checkpoint_monitor", "batch_size", "reduce_lr_factor", "reduce_lr_patience",
              "use_cutmix", "use_mixup", "cutmix_alpha", "mixup_alpha")
SUPERSEDED_DIR = os.path.join(CONFIG_BASE["output_dir"], f"{CONFIG_BASE['experiment_prefix']}_superseded")


def expected_settings(name):
    """The values apply_component(name) would put into CONFIG for STALE_KEYS."""
    p = COMPONENTS[name]
    exp = {k: CONFIG_BASE[k] for k in STALE_KEYS if k in CONFIG_BASE}
    exp["models_to_train"] = p["model"]
    exp.update({k: p[k] for k in _PRESET_TRAINING_KEYS if k in p})
    return exp


def saved_settings(name):
    """Settings recorded in the component's config.txt (None if not available)."""
    path = os.path.join(CONFIG_BASE["output_dir"], f"{CONFIG_BASE['experiment_prefix']}_{name}", "config.txt")
    if not os.path.exists(path):
        return None
    out = {}
    for line in open(path):
        if line.startswith("  ") and ":" in line:
            k, v = line.strip().split(":", 1)
            out[k.strip()] = v.strip()
    return out


def stale_components(order):
    stale = {}
    for name in order:
        if not os.path.exists(progress_path(name)):
            continue
        saved = saved_settings(name)
        if saved is None:
            print(f"[Stale] {name}: results exist but config.txt is missing — cannot verify, left as is")
            continue
        diffs = {k: (saved.get(k), str(v)) for k, v in expected_settings(name).items() if saved.get(k) != str(v)}
        if diffs:
            stale[name] = diffs
    return stale


def archive_stale(stale, state):
    """Move stale component results and ALL pipeline evaluations to SUPERSEDED_DIR; record it in the run state."""
    stamp = time.strftime("%Y%m%d_%H%M%S")
    for name, diffs in stale.items():
        src = os.path.join(CONFIG_BASE["output_dir"], f"{CONFIG_BASE['experiment_prefix']}_{name}")
        dst = os.path.join(SUPERSEDED_DIR, f"{name}_{stamp}")
        os.makedirs(SUPERSEDED_DIR, exist_ok=True)
        shutil.move(src, dst)
        with open(os.path.join(dst, "SUPERSEDED.txt"), "w") as f:
            f.write(f"Archived {stamp}: settings changed (saved -> current):\n")
            for k, (old, new) in diffs.items():
                f.write(f"  {k}: {old} -> {new}\n")
        for key in [k for k in state["runs"] if k.startswith(f"{name}/")]:
            state.setdefault("superseded", {})[f"{key}@{stamp}"] = {**state["runs"].pop(key), "reason": "settings changed"}
        print(f"[Stale] {name}: settings changed {diffs} -> old results moved to {dst}")
    if os.path.isdir(PIPELINE_DIR) and os.listdir(PIPELINE_DIR):
        dst = os.path.join(SUPERSEDED_DIR, f"pipeline_{stamp}")
        shutil.move(PIPELINE_DIR, dst)
        print(f"[Stale] pipeline evaluations depend on every component -> moved to {dst}")
    for key in list(state.get("evaluations", {})):
        state.setdefault("superseded", {})[f"pipeline/{key}@{stamp}"] = state["evaluations"].pop(key)
    _save_state(state)

def run_all(seeds, order, stop_on_error=False, evaluate=True):
    state = json.load(open(RUN_ALL_STATE)) if os.path.exists(RUN_ALL_STATE) else {"runs": {}, "evaluations": {}}
    unknown = [c for c in order if c not in COMPONENTS]
    if unknown:
        raise KeyError(f"Unknown components in order: {unknown}")
    stale = stale_components(order)
    if stale:
        archive_stale(stale, state)
    todo = [(s, c) for s in seeds for c in order if not seed_done(c, s)]
    print(f"[RunAll] {len(seeds) * len(order)} runs requested | already done: {len(seeds) * len(order) - len(todo)} "
          f"| to run: {len(todo)}")
    k = 0
    for seed in seeds:
        for comp in order:
            key = f"{comp}/seed{seed}"
            if seed_done(comp, seed):
                continue
            k += 1
            print(f"\n{'#' * 78}\n# [RunAll] {k}/{len(todo)}: {key}   ({datetime.now():%Y-%m-%d %H:%M})\n{'#' * 78}")
            free_memory()
            start = time.time()
            try:
                apply_component(comp, multi_seed_list=[seed])
                if not check_protocol():
                    raise RuntimeError("check_protocol() failed")
                run_multi_seed_experiment(CONFIG["models_to_train"])
                state["runs"][key] = {"status": "done", "minutes": round((time.time() - start) / 60, 1),
                                      "finished": f"{datetime.now():%Y-%m-%d %H:%M}"}
            except Exception as e:
                state["runs"][key] = {"status": "failed", "error": f"{type(e).__name__}: {e}"[:500],
                                      "traceback": traceback.format_exc()[-3000:],
                                      "at": f"{datetime.now():%Y-%m-%d %H:%M}"}
                print(f"\n[RunAll] FAILED {key}: {type(e).__name__}: {str(e)[:300]}")
                if stop_on_error:
                    _save_state(state)
                    raise
            finally:
                _save_state(state)
                free_memory()

        if evaluate:
            if not all(seed_done(c, seed) for c in COMPONENTS):
                print(f"\n[RunAll] seed {seed}: not all components finished — pipeline evaluation skipped")
            elif not pipeline_done(seed):
                try:
                    df, _ = evaluate_pipeline(seed)
                    theoretical_pipeline(seed, df)
                    state["evaluations"][f"seed{seed}"] = "done"
                except Exception as e:
                    state["evaluations"][f"seed{seed}"] = f"failed: {type(e).__name__}: {e}"[:500]
                    print(f"\n[RunAll] pipeline evaluation FAILED for seed {seed}: {e}")
                    if stop_on_error:
                        _save_state(state)
                        raise
                finally:
                    _save_state(state)
                    free_memory()

    # ── per-component summaries over every seed that is complete ─────────────────
    print(f"\n{'=' * 78}\n[RunAll] PER-COMPONENT SUMMARY\n{'=' * 78}")
    for comp in order:
        done = [s for s in seeds if seed_done(comp, s)]
        if len(done) == len(seeds):
            apply_component(comp, multi_seed_list=list(seeds))
            save_multi_seed_results(run_multi_seed_experiment(CONFIG["models_to_train"]))   # no training: all seeds done
        else:
            print(f"  {comp}: {len(done)}/{len(seeds)} seeds done — summary after the rest finish")
    evaluated = [s for s in seeds if pipeline_done(s)]
    if len(evaluated) > 1:
        print(f"\n[RunAll] ASSEMBLED PIPELINE over seeds {evaluated}")
        summarize_pipeline(evaluated)

    failed = {k: v["error"] for k, v in state["runs"].items() if v["status"] == "failed" and
              not seed_done(k.split("/")[0], int(k.split("seed")[1]))}
    remaining = [f"{c}/seed{s}" for s in seeds for c in order if not seed_done(c, s)]
    print(f"\n{'=' * 78}\n[RunAll] FINISHED | remaining: {len(remaining)} | failed: {len(failed)}\n{'=' * 78}")
    for k, v in failed.items():
        print(f"  FAILED {k}: {v[:200]}")
    if remaining:
        print("  Re-run the run-all cell to retry the remaining runs (finished runs are skipped).")
        print("  If a run keeps failing with an OOM: restart the kernel, run sections 1-9a, then the run-all cell.")
    return state


print("Run-all orchestrator defined.")


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# 9b. RUN EVERYTHING — safe to re-run at any time; finished runs are skipped.
# ═══════════════════════════════════════════════════════════════════════════════
RUN_SEEDS = CONFIG_BASE["multi_seed_list"]      # [42, 123, 2024, 779, 1027]
RUN_ORDER = ["Classifier", "Apple", "Bell_pepper", "Corn", "Potato", "Tomato", "Grape", "Rest"]
STOP_ON_ERROR = False                           # True: stop at the first failure instead of moving on
EVALUATE_EACH_SEED = True                       # evaluate the assembled pipeline once a seed is complete

run_all_state = run_all(RUN_SEEDS, RUN_ORDER, stop_on_error=STOP_ON_ERROR, evaluate=EVALUATE_EACH_SEED)


## 10. Optional — manual runs
One component by hand, or re-evaluate chosen seeds.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# 10. OPTIONAL — one component by hand (debugging, smoke tests). Shares state with run-all.
# ═══════════════════════════════════════════════════════════════════════════════
COMPONENT = "Classifier"
SEEDS = [42]

apply_component(COMPONENT, multi_seed_list=SEEDS)          # e.g. apply_component("Tomato", multi_seed_list=[42], epochs=5)
assert check_protocol(), "Protocol check failed — see the ! lines above."
results = run_multi_seed_experiment(CONFIG["models_to_train"])
if results["complete"]:
    save_multi_seed_results(results)


In [ ]:
# Run after all 8 components are trained for the seed(s) below.
EVAL_SEEDS = [42]

for s in EVAL_SEEDS:
    pipeline_df, _ = evaluate_pipeline(s)
    theoretical_pipeline(s, pipeline_df)
if len(EVAL_SEEDS) > 1:
    summarize_pipeline(EVAL_SEEDS)


## 11. Seed ensemble + test-time augmentation (no training)
Uses the checkpoints already trained for all 5 seeds. Each component predicts every image of a split once per seed
(and once mirrored, for flip TTA); outputs are cached in `<prefix>_pipeline/probs/`. To avoid out-of-memory errors, every model runs in its own process (11a-2), with checkpoints and automatic retry at a smaller batch size, so every variant below takes seconds.

**Protocol:** compare variants on the **validation** split only, fix the choice, then evaluate that one variant on test once.
Report it as a 5-model ensemble (5× inference cost) next to the single-model mean ± CI — never instead of it.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# 11a. SEED ENSEMBLE + TTA (definitions) — no training
#   ensemble  : average the softmax outputs of the 5 seeds (classifier AND specialists)
#   tta       : average each model's output on the image and its horizontal mirror
#               (valid: RandomFlip("horizontal") is part of training augmentation)
#   routing   : "hard" = route to the classifier's top species (as in section 8)
#               "soft" = score(class) = P(species) x P(class | species), over all species
# ═══════════════════════════════════════════════════════════════════════════════
import itertools
import json
import gc

PROB_DIR = os.path.join(PIPELINE_DIR, "probs")
ALLOW_INPROCESS_CACHE = False   # True = compute missing predictions inside the notebook (can OOM)
CLASSIFIER = next(n for n, p in COMPONENTS.items() if p["type"] == "classifier")


def split_table(split):
    d = final_split_dirs()[split]
    return pd.DataFrame([{"image": os.path.join(d, c, f), "true_class": c, "true_species": species_of(c)}
                         for c in _class_dirs(d) for f in sorted(os.listdir(os.path.join(d, c)))
                         if f.lower().endswith(TF_IMAGE_EXT)])


def component_probs(name, seed, split, tta=False, batch_size=8):
    """Softmax outputs of one component/seed on EVERY image of `split` (cached to disk)."""
    os.makedirs(PROB_DIR, exist_ok=True)
    path = os.path.join(PROB_DIR, f"{split}_{name}_seed{seed}{'_flip' if tta else ''}.npz")
    paths = list(split_table(split)["image"])
    names = component_class_names(name)
    if os.path.exists(path):
        z = np.load(path, allow_pickle=False)
        if list(z["paths"]) == paths and list(z["classes"]) == names:
            return z["probs"], names
        print(f"  [cache] {os.path.basename(path)} does not match the current split — recomputing")
    if not ALLOW_INPROCESS_CACHE:
        raise RuntimeError(f"No cached predictions for {split} {name} seed {seed}{' +flip' if tta else ''}. "
                           "Run write_cache_jobs() and run_cache_jobs() (cell 11a-2) first.")
    plain = component_probs(name, seed, split, tta=False, batch_size=batch_size)[0] if tta else None
    ckpt = component_checkpoint(name, seed)
    if not os.path.exists(ckpt):
        raise RuntimeError(f"No checkpoint for {name} seed {seed}: {ckpt}")
    p = COMPONENTS[name]
    model = tf.keras.models.load_model(ckpt)
    if model.output_shape[-1] != len(names):
        raise RuntimeError(f"{name}: model has {model.output_shape[-1]} outputs but {len(names)} classes")
    prep = _make_preprocess_fn(p["model"])
    out = []
    for i in range(0, len(paths), batch_size):
        batch = np.stack([_load_tf_image(q, p["img_size"]) for q in paths[i:i + batch_size]])
        if tta:
            batch = np.ascontiguousarray(batch[:, :, ::-1, :])      # horizontal mirror
        out.append(np.asarray(model(prep(batch), training=False)))
    del model
    tf.keras.backend.clear_session()
    gc.collect()
    probs = np.concatenate(out)
    if tta:
        probs = (plain + probs) / 2
    np.savez(path, probs=probs, classes=np.array(names), paths=np.array(paths))
    print(f"  [cache] {split} {name} seed {seed}{' +flip' if tta else ''}: {len(paths)} images")
    return probs, names


def cache_all(split, seeds, tta=True):
    """Fill the cache for every component and seed (tta=True fills plain AND flip)."""
    for s in seeds:
        for name in COMPONENTS:
            component_probs(name, s, split, tta=tta)


def _mean_probs(name, seeds, split, tta):
    return np.mean([component_probs(name, s, split, tta)[0] for s in seeds], axis=0), component_class_names(name)


def pipeline_from_probs(split, seeds, tta=False, routing="hard"):
    """Assembled pipeline from cached outputs; len(seeds) == 1 reproduces section 8."""
    df = split_table(split)
    P, species = _mean_probs(CLASSIFIER, seeds, split, tta)
    spec = {n: _mean_probs(n, seeds, split, tta) for n in COMPONENTS if n != CLASSIFIER}
    if set(species) != set(spec):
        raise RuntimeError(f"Classifier outputs {species} but components are {list(spec)}")
    df["routed_to"] = np.array(species)[P.argmax(1)]
    df["classifier_confidence"] = P.max(1)
    df["routing_correct"] = df["routed_to"] == df["true_species"]
    if routing == "hard":
        df["predicted_class"] = [spec[r][1][int(spec[r][0][i].argmax())] for i, r in enumerate(df["routed_to"])]
    elif routing == "soft":
        scores, classes = [], []
        for j, s in enumerate(species):
            q, cls = spec[s]
            scores.append(P[:, j:j + 1] * q)
            classes += cls
        df["predicted_class"] = np.array(classes)[np.concatenate(scores, axis=1).argmax(1)]
    else:
        raise ValueError(routing)
    df["final_correct"] = df["predicted_class"] == df["true_class"]
    return df


def bootstrap_ci(correct, level=0.95, n_boot=10000, seed=0):
    """Percentile bootstrap over test images (the ensemble is ONE system, so no across-seed CI)."""
    c = np.asarray(correct, dtype=float)
    means = np.random.default_rng(seed).choice(c, size=(n_boot, len(c))).mean(axis=1)
    return tuple(np.percentile(means, [50 * (1 - level), 50 * (1 + level)]))


def check_matches_section8(seeds):
    """Single-seed, hard, no-TTA test results must equal section 8's saved results."""
    for s in seeds:
        f = os.path.join(PIPELINE_DIR, f"pipeline_summary_seed{s}.json")
        if not os.path.exists(f):
            print(f"  seed {s}: no section-8 summary to compare")
            continue
        got = int(pipeline_from_probs("test", [s])["final_correct"].sum())
        want = json.load(open(f))["final_correct"]
        print(f"  seed {s}: cache {got} vs section 8 {want} -> {'OK' if got == want else 'MISMATCH — stop and check'}")


def compare_variants(split, seeds):
    """Single-seed mean vs 5-seed ensemble for every variant. Use split='val' to choose."""
    rows = []
    for tta in (False, True):
        for routing in ("hard", "soft"):
            single = [pipeline_from_probs(split, [s], tta, routing)["final_correct"].mean() for s in seeds]
            ens = pipeline_from_probs(split, seeds, tta, routing)
            rows.append({"tta": tta, "routing": routing,
                         "single_mean": np.mean(single), "single_std": np.std(single, ddof=1),
                         "ensemble": ens["final_correct"].mean(), "ensemble_routing": ens["routing_correct"].mean(),
                         "ensemble_correct": int(ens["final_correct"].sum()), "n": len(ens)})
    out = pd.DataFrame(rows)
    print(f"[{split}] {len(seeds)} seeds")
    print((out.assign(**{c: 100 * out[c] for c in ("single_mean", "single_std", "ensemble", "ensemble_routing")})
              .round(2).to_string(index=False)))
    out.to_csv(os.path.join(PIPELINE_DIR, f"ensemble_variants_{split}.csv"), index=False)
    return out


def ensemble_size_curve(split, seeds, tta=False, routing="hard"):
    """Accuracy of every k-seed subset, k = 1..len(seeds)."""
    rows = []
    for k in range(1, len(seeds) + 1):
        accs = [pipeline_from_probs(split, list(c), tta, routing)["final_correct"].mean()
                for c in itertools.combinations(seeds, k)]
        rows.append({"k": k, "subsets": len(accs), "mean": np.mean(accs), "min": np.min(accs), "max": np.max(accs)})
    out = pd.DataFrame(rows)
    print(out.assign(**{c: (100 * out[c]).round(2) for c in ("mean", "min", "max")}).to_string(index=False))
    out.to_csv(os.path.join(PIPELINE_DIR, f"ensemble_size_curve_{split}.csv"), index=False)
    return out


def report_final(seeds, tta, routing):
    """The ONE test evaluation of the chosen variant."""
    df = pipeline_from_probs("test", seeds, tta, routing)
    acc, (lo, hi) = df["final_correct"].mean(), bootstrap_ci(df["final_correct"])
    single = [pipeline_from_probs("test", [s], tta=False, routing="hard")["final_correct"].mean() for s in seeds]
    print(f"Single model (section 8): {100 * np.mean(single):.2f}% ± {100 * np.std(single, ddof=1):.2f}% over {len(seeds)} seeds")
    print(f"Ensemble ({len(seeds)} seeds, tta={tta}, routing={routing}): "
          f"{int(df['final_correct'].sum())}/{len(df)} = {100 * acc:.2f}%  (95% bootstrap CI [{100 * lo:.2f}%, {100 * hi:.2f}%])")
    print(f"Ensemble routing accuracy: {100 * df['routing_correct'].mean():.2f}%")
    per = df.groupby("true_species").agg(images=("image", "size"), routed_ok=("routing_correct", "sum"),
                                         correct=("final_correct", "sum"))
    per["end_to_end_acc"] = (per["correct"] / per["images"]).round(4)
    print(per.to_string())
    tag = f"ensemble{len(seeds)}_{'flip' if tta else 'plain'}_{routing}"
    df.to_csv(os.path.join(PIPELINE_DIR, f"{tag}_test_log.csv"), index=False)
    json.dump({"seeds": list(seeds), "tta": tta, "routing": routing, "n": len(df),
               "final_correct": int(df["final_correct"].sum()), "final_accuracy": float(acc),
               "ci_low": float(lo), "ci_high": float(hi), "routing_accuracy": float(df["routing_correct"].mean()),
               "single_model_mean": float(np.mean(single)), "single_model_std": float(np.std(single, ddof=1))},
              open(os.path.join(PIPELINE_DIR, f"{tag}_test_summary.json"), "w"), indent=2)
    return df


print("Ensemble/TTA evaluation defined.")


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# 11a-2. OOM-SAFE CACHING — every model runs in its OWN Python process
#   TensorFlow never fully returns GPU memory inside one process, so loading 80 large
#   models in the notebook eventually runs out. Here each (component, seed, split, flip)
#   job runs in a fresh subprocess that exits when done, freeing all GPU/CPU memory.
#   Checkpoints:
#     - a finished job is saved as its .npz and skipped on every later run
#     - inside a job, predictions are saved every `save_every` batches (.partial.npz)
#       and a crashed job resumes from there
#     - on a crash (OOM), the job is retried with half the batch size, down to 1
#   Output files are the same ones component_probs() reads, so 11a works unchanged.
# ═══════════════════════════════════════════════════════════════════════════════
import subprocess
import time
import sys

CACHE_JOBS = os.path.join(PROB_DIR, "cache_jobs.json")
CACHE_WORKER = os.path.join(PROB_DIR, "cache_worker.py")
CACHE_LOG = os.path.join(PROB_DIR, "cache_log.txt")

_WORKER_SRC = r'''
import os, sys, json
os.environ.setdefault("KERAS_BACKEND", "tensorflow")
os.environ["TF_FORCE_GPU_ALLOW_GROWTH"] = "true"
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")
import numpy as np
import tensorflow as tf
for g in tf.config.list_physical_devices("GPU"):
    try:
        tf.config.experimental.set_memory_growth(g, True)
    except Exception:
        pass

PREP = {"convnext": tf.keras.applications.convnext.preprocess_input,
        "efficientnet": tf.keras.applications.efficientnet.preprocess_input,
        "efficientnet_v2": tf.keras.applications.efficientnet_v2.preprocess_input}


def load_image(path, size):          # identical to _load_tf_image in section 8
    img = tf.image.decode_image(tf.io.read_file(path), channels=3, expand_animations=False)
    return tf.image.resize(img, [size, size], method="bilinear").numpy().astype(np.float32)


def save(path, **arrays):            # atomic: a crash never leaves a half-written file
    tmp = path + ".tmp"
    with open(tmp, "wb") as f:
        np.savez(f, **arrays)
    os.replace(tmp, path)


def main(jobs_file, idx, batch_size, save_every):
    job = json.load(open(jobs_file))[idx]
    for mod in job.get("custom_imports", []):
        __import__(mod)
    paths, out = job["paths"], job["out"]
    partial = out + ".partial.npz"
    done = []
    if os.path.exists(partial):
        z = np.load(partial, allow_pickle=False)
        if list(z["paths"]) == paths:
            done = list(z["probs"])
            print(f"    resuming from image {len(done)}/{len(paths)}", flush=True)
    model = tf.keras.models.load_model(job["ckpt"])
    if model.output_shape[-1] != len(job["classes"]):
        raise RuntimeError(f"model has {model.output_shape[-1]} outputs, expected {len(job['classes'])}")
    prep = PREP[job["prep"]]
    n_batches = 0
    for i in range(len(done), len(paths), batch_size):
        batch = np.stack([load_image(p, job["img_size"]) for p in paths[i:i + batch_size]])
        if job["flip"]:
            batch = np.ascontiguousarray(batch[:, :, ::-1, :])
        done.extend(np.asarray(model(prep(batch), training=False)))
        n_batches += 1
        if n_batches % save_every == 0:
            save(partial, probs=np.stack(done), paths=np.array(paths))
    probs = np.stack(done)
    if job["flip"]:
        z = np.load(job["plain_out"], allow_pickle=False)
        if list(z["paths"]) != paths:
            raise RuntimeError("plain cache does not match this job's images")
        probs = (z["probs"] + probs) / 2
    save(out, probs=probs, classes=np.array(job["classes"]), paths=np.array(paths))
    if os.path.exists(partial):
        os.remove(partial)
    print(f"    saved {os.path.basename(out)} ({len(paths)} images)", flush=True)


if __name__ == "__main__":
    main(sys.argv[1], int(sys.argv[2]), int(sys.argv[3]), int(sys.argv[4]))
'''


def _prep_key(model_name):
    fn = _make_preprocess_fn(model_name)
    for key, ref in (("convnext", tf.keras.applications.convnext.preprocess_input),
                     ("efficientnet", tf.keras.applications.efficientnet.preprocess_input),
                     ("efficientnet_v2", tf.keras.applications.efficientnet_v2.preprocess_input)):
        if fn is ref:
            return key
    raise RuntimeError(f"Worker has no preprocessing matching '{model_name}' — add it to PREP in _WORKER_SRC")


def _cache_file(split, name, seed, flip):
    return os.path.join(PROB_DIR, f"{split}_{name}_seed{seed}{'_flip' if flip else ''}.npz")


def _cache_ok(path, paths, classes):
    if not os.path.exists(path):
        return False
    try:
        z = np.load(path, allow_pickle=False)
        return list(z["paths"]) == paths and list(z["classes"]) == classes
    except Exception:
        return False


def _job_paths(split, name):
    """Images a job predicts: the whole split, except on TRAIN where a specialist only
    sees its own classes (teacher predictions for the student; avoids 5x wasted work)."""
    df = split_table(split)
    if split == "train" and COMPONENTS[name]["type"] != "classifier":
        df = df[df["true_class"].isin(component_class_names(name))]
    return list(df["image"])


def write_cache_jobs(splits=("val", "test"), seeds=None, tta=True, custom_imports=(), flip_splits=("val", "test")):
    """Write the job list (plain jobs before their flip jobs) and the worker script.
    Flip TTA is only computed for splits in flip_splits (never needed for train)."""
    seeds = seeds or CONFIG_BASE["multi_seed_list"]
    os.makedirs(PROB_DIR, exist_ok=True)
    jobs = []
    for split in splits:
        for seed in seeds:
            for name, p in COMPONENTS.items():
                paths = _job_paths(split, name)
                ckpt = component_checkpoint(name, seed)
                if not os.path.exists(ckpt):
                    raise RuntimeError(f"No checkpoint for {name} seed {seed}: {ckpt}")
                common = {"name": name, "seed": seed, "split": split, "ckpt": ckpt,
                          "img_size": p["img_size"], "prep": _prep_key(p["model"]),
                          "classes": component_class_names(name), "paths": paths,
                          "custom_imports": list(custom_imports)}
                jobs.append({**common, "flip": False, "out": _cache_file(split, name, seed, False)})
                if tta and split in flip_splits:
                    jobs.append({**common, "flip": True, "out": _cache_file(split, name, seed, True),
                                 "plain_out": _cache_file(split, name, seed, False)})
    json.dump(jobs, open(CACHE_JOBS, "w"))
    open(CACHE_WORKER, "w").write(_WORKER_SRC)
    todo = sum(not _cache_ok(j["out"], j["paths"], j["classes"]) for j in jobs)
    print(f"[Cache] {len(jobs)} jobs written, {todo} still to run -> {CACHE_JOBS}")
    return jobs


def run_cache_jobs(batch_size=8, save_every=5, max_failures=3):
    """Run every unfinished job in its own process. Safe to stop and re-run at any time."""
    jobs = json.load(open(CACHE_JOBS))
    env = {**os.environ, "PYTHONPATH": os.pathsep.join([os.getcwd(), os.environ.get("PYTHONPATH", "")])}
    failed = []
    for idx, job in enumerate(jobs):
        label = f"{job['split']} {job['name']} seed {job['seed']}{' +flip' if job['flip'] else ''}"
        if _cache_ok(job["out"], job["paths"], job["classes"]):
            continue
        if job["flip"] and not _cache_ok(job["plain_out"], job["paths"], job["classes"]):
            print(f"[Cache] {idx + 1}/{len(jobs)} {label}: skipped (plain run not finished)")
            failed.append(label)
            continue
        bs, failures = batch_size, 0
        while True:
            print(f"[Cache] {idx + 1}/{len(jobs)} {label} (batch {bs})", flush=True)
            t0 = time.time()
            proc = subprocess.Popen([sys.executable, CACHE_WORKER, CACHE_JOBS, str(idx), str(bs), str(save_every)],
                                    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=env)
            tail = []
            for line in proc.stdout:
                tail = (tail + [line.rstrip()])[-15:]
                if line.startswith(("    resuming", "    saved")):
                    print(line.rstrip(), flush=True)
            proc.wait()
            with open(CACHE_LOG, "a") as log:
                log.write(f"{time.strftime('%Y-%m-%d %H:%M:%S')} | {label} | batch {bs} | "
                          f"exit {proc.returncode} | {time.time() - t0:.0f}s\n")
            if proc.returncode == 0:
                break
            failures += 1
            oom = any(k in "\n".join(tail) for k in ("OOM", "ResourceExhausted", "out of memory", "Killed"))
            print(f"    FAILED (exit {proc.returncode}{', out of memory' if oom else ''}). Last lines:")
            print("\n".join("      " + t for t in tail[-6:]))
            if bs > 1:
                bs = max(1, bs // 2)          # partial progress is kept; retry smaller
            elif failures >= max_failures:
                print(f"    giving up on {label} — see {CACHE_LOG}")
                failed.append(label)
                break
    done = sum(_cache_ok(j["out"], j["paths"], j["classes"]) for j in jobs)
    print(f"\n[Cache] {done}/{len(jobs)} jobs complete." + (f" Failed: {failed}" if failed else ""))
    return not failed


print("OOM-safe caching defined.")


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# 11b. RUN — step by step. Safe to stop and re-run at any point (OOM, crash, restart):
#       finished predictions are skipped and a crashed model resumes from its last save.
# ═══════════════════════════════════════════════════════════════════════════════
SEEDS = CONFIG_BASE["multi_seed_list"]

# Step 1 — cache predictions: 8 components x 5 seeds x 2 splits x (plain + flip) = 160 jobs,
#          each in its own process. Re-run this cell until it prints "160/160 jobs complete".
#          (Teacher predictions on TRAIN are a separate job list — see section 12.)
write_cache_jobs(splits=("val", "test"), seeds=SEEDS, tta=True)
all_cached = run_cache_jobs(batch_size=8, save_every=5)

# Step 2 — only once everything is cached
if all_cached:
    check_matches_section8(SEEDS)        # every seed must print OK
    val_variants = compare_variants("val", SEEDS)    # choose on VALIDATION only
    ensemble_size_curve("val", SEEDS)
else:
    print("Not all predictions are cached yet — re-run this cell.")


In [ ]:
# Step 3 — set CHOSEN from the validation table above, then run this cell ONCE.
CHOSEN = {"tta": False, "routing": "hard"}     # <- fill in from Step 2
final_df = report_final(SEEDS, **CHOSEN)
ensemble_size_curve("test", SEEDS, **CHOSEN)   # for the paper figure: accuracy vs number of seeds


## 12. Teacher → student (step 1: what do the teachers say about the training images?)
All v2 runs so far are the **teachers**. Before training a student we check two things on the TRAINING images only:
1. **Did the teachers memorise their training set?** If the 5-seed ensemble gives the given label ~100% on every image,
   its soft labels add little and label-error detection needs out-of-fold predictions instead.
2. **Which training images do the teachers reject?** An image that all 5 seeds still disagree with, after being trained on it,
   is a strong mislabel candidate. These go to a review CSV (y/n), like the near-duplicate review.

Validation and test are never used here. The student itself (soft teacher labels ± per-image weights) is built after this check.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# 12a. TEACHER DIAGNOSTIC (definitions) — training images only
# ═══════════════════════════════════════════════════════════════════════════════
TEACHER_DIR = os.path.join(CONFIG_BASE["output_dir"], f"{CONFIG_BASE['experiment_prefix']}_teacher")


def teacher_probs(name, seeds=None):
    """(seeds, images, classes) teacher outputs on this component's own TRAIN images."""
    seeds = seeds or CONFIG_BASE["multi_seed_list"]
    paths, classes = _job_paths("train", name), component_class_names(name)
    out = []
    for s in seeds:
        f = _cache_file("train", name, s, False)
        if not _cache_ok(f, paths, classes):
            raise RuntimeError(f"No train predictions for {name} seed {s}. Run the 12b caching step first.")
        out.append(np.load(f, allow_pickle=False)["probs"])
    return np.stack(out), paths, classes


def teacher_report(name, seeds=None, suspect_min_seeds=3):
    seeds = seeds or CONFIG_BASE["multi_seed_list"]
    P, paths, classes = teacher_probs(name, seeds)
    suspect_min_seeds = min(suspect_min_seeds, len(seeds))
    df = split_table("train").set_index("image").loc[paths].reset_index()
    given = df["true_species"] if COMPONENTS[name]["type"] == "classifier" else df["true_class"]
    gi = np.array([classes.index(g) for g in given])
    mean = P.mean(axis=0)
    rows = np.arange(len(df))
    df["given_label"] = given.values
    df["teacher_pred"] = np.array(classes)[mean.argmax(1)]
    df["p_given"] = mean[rows, gi]
    df["p_teacher_pred"] = mean.max(1)
    df["seeds_disagreeing"] = (P.argmax(2) != gi).sum(axis=0)
    df["p_given_seed_std"] = P[:, rows, gi].std(axis=0)

    n = len(df)
    acc = (df["teacher_pred"] == df["given_label"]).mean()
    print(f"\n{'=' * 70}\n  TEACHER on TRAIN — {name} ({n} images, {len(seeds)} seeds)\n{'=' * 70}")
    print(f"  ensemble train accuracy       : {100 * acc:.2f}%")
    print(f"  mean / median p(given label)  : {df['p_given'].mean():.3f} / {df['p_given'].median():.3f}")
    print(f"  images with p(given) < 0.5    : {(df['p_given'] < 0.5).sum()}")
    for k in range(len(seeds), 0, -1):
        print(f"  rejected by >= {k} of {len(seeds)} seeds  : {(df['seeds_disagreeing'] >= k).sum()}")
    per = df.groupby("given_label").agg(images=("image", "size"),
                                        teacher_acc=("teacher_pred", lambda s: 0.0),
                                        mean_p_given=("p_given", "mean"),
                                        rejected_by_all=("seeds_disagreeing", lambda s: int((s == len(seeds)).sum())))
    per["teacher_acc"] = df.assign(ok=df["teacher_pred"] == df["given_label"]).groupby("given_label")["ok"].mean()
    print("\n" + per.round(3).to_string())

    if df["p_given"].mean() > 0.95 and (df["seeds_disagreeing"] >= suspect_min_seeds).sum() <= max(1, 0.01 * n):
        verdict = ("MEMORISED — teachers fit almost every training image. Soft labels add little; "
                   "label-error scores need out-of-fold predictions.")
    else:
        verdict = ("NOT MEMORISED — teachers still disagree with some training labels. Soft labels carry "
                   "information and the suspects below are worth reviewing.")
    print(f"\n  VERDICT: {verdict}")

    os.makedirs(TEACHER_DIR, exist_ok=True)
    df.to_csv(os.path.join(TEACHER_DIR, f"{name}_teacher_train.csv"), index=False)
    sus = (df[df["seeds_disagreeing"] >= suspect_min_seeds]
           .sort_values(["seeds_disagreeing", "p_given"], ascending=[False, True]).copy())
    sus["is_mislabeled (y/n)"] = ""
    sus["correct_label_if_known"] = ""
    sus_path = os.path.join(TEACHER_DIR, f"{name}_suspects_for_review.csv")
    sus.to_csv(sus_path, index=False)
    print(f"  {len(sus)} suspect(s) -> {sus_path}")
    return {"component": name, "n": n, "train_acc": float(acc), "mean_p_given": float(df["p_given"].mean()),
            "p_given_below_05": int((df["p_given"] < 0.5).sum()),
            "rejected_by_all": int((df["seeds_disagreeing"] == len(seeds)).sum()),
            "suspects": len(sus), "verdict": verdict.split(" —")[0]}


def teacher_summary(seeds=None):
    rows = [teacher_report(name, seeds) for name in COMPONENTS]
    out = pd.DataFrame(rows)
    print("\n" + out.drop(columns="verdict").round(4).to_string(index=False))
    print(out[["component", "verdict"]].to_string(index=False))
    out.to_csv(os.path.join(TEACHER_DIR, "teacher_summary.csv"), index=False)
    return out


print("Teacher diagnostic defined.")


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# 12b. RUN — teacher predictions on TRAIN (8 components x 5 seeds = 40 jobs, no flip),
#       same OOM-safe one-process-per-model caching as 11. Re-run until all jobs complete.
# ═══════════════════════════════════════════════════════════════════════════════
SEEDS = CONFIG_BASE["multi_seed_list"]
write_cache_jobs(splits=("train",), seeds=SEEDS, tta=False)
if run_cache_jobs(batch_size=8, save_every=5):
    teacher_table = teacher_summary(SEEDS)
else:
    print("Not all train predictions are cached yet — re-run this cell.")


## 13. Label quality — 5-fold out-of-fold scores (proof of concept for the student)
Every training image is scored by a model that never saw it: frozen ImageNet ConvNeXt-Large features → 5-fold
StratifiedGroupKFold (look-alike pairs kept in one fold) → logistic regression → cleanlab label-quality score.

**Outputs** (`<prefix>_label_quality/`): per-component score CSV, a review CSV of flagged images with a
`is_mislabeled (y/n)` column, contact sheets (given vs predicted label), and the **proof of concept**:
the same linear probe trained on (A) all labels, (B) quality-weighted labels, (C) flagged images removed — judged on
validation only. If B or C wins and your review confirms real label errors, we spend GPU time on the full student.

Needs `pip install cleanlab` once (otherwise a simpler self-confidence fallback is used). Sections 11–12 are not required.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# 13a. LABEL-QUALITY SCORES — 5-fold out-of-fold (proof of concept, no GPU training)
#   1. Frozen ImageNet backbone -> one feature vector per TRAIN and VAL image
#      (one process per split, checkpointed, OOM-safe — same scheme as 11a-2).
#   2. Per component: 5-fold StratifiedGroupKFold on TRAIN (groups = reviewed near-duplicate
#      pairs, like the validation split) -> logistic regression -> every training image is
#      predicted by a model that never saw it.
#   3. cleanlab turns those predictions into a label-quality score (0..1) + a list of likely
#      label issues -> review CSV (y/n) + contact sheets.
#   4. Proof of concept on VALIDATION: does a linear probe trained with weighted / cleaned
#      labels beat the same probe trained on all labels?  (Test is never touched.)
# ═══════════════════════════════════════════════════════════════════════════════
import subprocess
import sys
import time
import json
import gc
import cv2
from collections import defaultdict
from PIL import Image, ImageDraw
from sklearn.linear_model import LogisticRegression, LogisticRegressionCV
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import log_loss

LQ = {
    "backbone": "ConvNeXtLarge",          # any tf.keras.applications ConvNeXt* (frozen, ImageNet)
    "img_size": 384,
    "n_folds": 5,
    "fold_seed": 42,
    "Cs": [0.001, 0.01, 0.1, 1.0],        # logistic-regression regularisation grid (inner 3-fold CV)
    "weight_floor": 0.2,                  # student weight = clip(label_quality, floor, 1)
    "near_dup_review_csv": os.path.join(os.path.dirname(CONFIG_BASE["dataset_root"]),
                                        "reports", "near_duplicate_review.csv"),
    "out_dir": os.path.join(CONFIG_BASE["output_dir"], f"{CONFIG_BASE['experiment_prefix']}_label_quality"),
}
LQ_JOBS = os.path.join(LQ["out_dir"], "feature_jobs.json")
LQ_WORKER = os.path.join(LQ["out_dir"], "feature_worker.py")

_FEATURE_WORKER_SRC = r'''
import os, sys, json
os.environ.setdefault("KERAS_BACKEND", "tensorflow")
os.environ["TF_FORCE_GPU_ALLOW_GROWTH"] = "true"
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")
import numpy as np
import tensorflow as tf
for g in tf.config.list_physical_devices("GPU"):
    try:
        tf.config.experimental.set_memory_growth(g, True)
    except Exception:
        pass


def load_image(path, size):          # identical to _load_tf_image in section 8
    img = tf.image.decode_image(tf.io.read_file(path), channels=3, expand_animations=False)
    return tf.image.resize(img, [size, size], method="bilinear").numpy().astype(np.float32)


def save(path, **arrays):
    tmp = path + ".tmp"
    with open(tmp, "wb") as f:
        np.savez(f, **arrays)
    os.replace(tmp, path)


def main(jobs_file, idx, batch_size, save_every):
    job = json.load(open(jobs_file))[idx]
    paths, out, size = job["paths"], job["out"], job["img_size"]
    partial = out + ".partial.npz"
    feats = []
    if os.path.exists(partial):
        z = np.load(partial, allow_pickle=False)
        if list(z["paths"]) == paths:
            feats = list(z["features"])
            print(f"    resuming from image {len(feats)}/{len(paths)}", flush=True)
    ctor = getattr(tf.keras.applications, job["backbone"])
    model = ctor(include_top=False, weights="imagenet", pooling="avg", input_shape=(size, size, 3))
    n = 0
    for i in range(len(feats), len(paths), batch_size):
        x = np.stack([load_image(p, size) for p in paths[i:i + batch_size]])
        feats.extend(np.asarray(model(x, training=False)))     # ConvNeXt normalises internally
        n += 1
        if n % save_every == 0:
            save(partial, features=np.stack(feats), paths=np.array(paths))
            print(f"    {len(feats)}/{len(paths)}", flush=True)
    save(out, features=np.stack(feats), paths=np.array(paths))
    if os.path.exists(partial):
        os.remove(partial)
    print(f"    saved {os.path.basename(out)} ({len(paths)} images)", flush=True)


if __name__ == "__main__":
    main(sys.argv[1], int(sys.argv[2]), int(sys.argv[3]), int(sys.argv[4]))
'''


def _feature_file(split):
    return os.path.join(LQ["out_dir"], f"features_{split}_{LQ['backbone']}_{LQ['img_size']}.npz")


def _features_ok(split):
    f = _feature_file(split)
    if not os.path.exists(f):
        return False
    try:
        return list(np.load(f, allow_pickle=False)["paths"]) == list(split_table(split)["image"])
    except Exception:
        return False


def extract_features(splits=("train", "val"), batch_size=16, save_every=5, max_failures=3):
    """Frozen-backbone features, one subprocess per split, checkpointed. Re-run until complete."""
    if not LQ["backbone"].lower().startswith("convnext"):
        raise ValueError("Worker assumes a ConvNeXt backbone (built-in normalisation).")
    os.makedirs(LQ["out_dir"], exist_ok=True)
    jobs = [{"split": s, "paths": list(split_table(s)["image"]), "out": _feature_file(s),
             "backbone": LQ["backbone"], "img_size": LQ["img_size"]} for s in splits]
    json.dump(jobs, open(LQ_JOBS, "w"))
    open(LQ_WORKER, "w").write(_FEATURE_WORKER_SRC)
    for idx, job in enumerate(jobs):
        if _features_ok(job["split"]):
            print(f"[Features] {job['split']}: cached")
            continue
        bs, failures = batch_size, 0
        while True:
            print(f"[Features] {job['split']}: {len(job['paths'])} images (batch {bs})", flush=True)
            proc = subprocess.Popen([sys.executable, LQ_WORKER, LQ_JOBS, str(idx), str(bs), str(save_every)],
                                    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
            tail = []
            for line in proc.stdout:
                tail = (tail + [line.rstrip()])[-15:]
                if line.startswith("    ") and not line.startswith("    File"):
                    print(line.rstrip(), flush=True)
            proc.wait()
            if proc.returncode == 0:
                break
            failures += 1
            print(f"    FAILED (exit {proc.returncode}). Last lines:\n" + "\n".join("      " + t for t in tail[-6:]))
            if bs > 1:
                bs = max(1, bs // 2)
            elif failures >= max_failures:
                print("    giving up")
                return False
    return all(_features_ok(s) for s in splits)


def _load_features(split):
    if not _features_ok(split):
        raise RuntimeError(f"Features for {split} missing — run extract_features().")
    return np.load(_feature_file(split), allow_pickle=False)["features"]


def _component_rows(name, split):
    """This component's images in `split` and their given labels (species for the classifier)."""
    df = split_table(split).reset_index(drop=True)
    classes = component_class_names(name)
    if COMPONENTS[name]["type"] == "classifier":
        df["given_label"] = df["true_species"]
    else:
        df = df[df["true_class"].isin(classes)].copy()
        df["given_label"] = df["true_class"]
    df["row"] = df.index
    df["rel"] = df["true_class"] + "/" + df["image"].map(os.path.basename)
    df["y"] = df["given_label"].map({c: i for i, c in enumerate(classes)})
    return df.reset_index(drop=True), classes


def _near_dup_groups(rels):
    """Group ids that keep reviewed look-alike pairs (answered 'n') in the same fold."""
    parent = {r: r for r in rels}

    def find(a):
        while parent[a] != a:
            parent[a] = parent[parent[a]]
            a = parent[a]
        return a

    path, linked = LQ["near_dup_review_csv"], 0
    if os.path.exists(path):
        for r in pd.read_csv(path).itertuples():
            a, b = str(r.a_rel), str(r.b_rel)
            if str(r.is_duplicate).strip().lower() != "y" and a in parent and b in parent:
                parent[find(a)] = find(b)
                linked += 1
    else:
        print(f"  ! {path} not found — folds are not grouped (set LQ['near_dup_review_csv'])")
    roots = {r: find(r) for r in rels}
    ids = {g: i for i, g in enumerate(sorted(set(roots.values())))}
    return np.array([ids[roots[r]] for r in rels]), linked


def _predict_full(clf, X, n_classes):
    """predict_proba with a column for every class (a fold may miss a rare class)."""
    p = np.zeros((len(X), n_classes))
    p[:, clf.classes_] = clf.predict_proba(X)
    return p


def _image_quality(paths):
    rows = []
    for p in paths:
        img = cv2.imread(p)
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        rows.append({"blur_laplacian_var": float(cv2.Laplacian(gray, cv2.CV_64F).var()),
                     "brightness": float(gray.mean()), "contrast": float(gray.std())})
    return pd.DataFrame(rows)


def contact_sheet(df, out_prefix, per_page=40, cols=5, tile=200):
    """Thumbnails of flagged images with given vs predicted label, for the y/n review."""
    pages = []
    for start in range(0, len(df), per_page):
        chunk = df.iloc[start:start + per_page]
        rows = (len(chunk) + cols - 1) // cols
        sheet = Image.new("RGB", (cols * tile, rows * (tile + 46)), "white")
        draw = ImageDraw.Draw(sheet)
        for k, r in enumerate(chunk.itertuples()):
            x, y = (k % cols) * tile, (k // cols) * (tile + 46)
            sheet.paste(Image.open(r.image).convert("RGB").resize((tile, tile)), (x, y))
            draw.text((x + 3, y + tile + 2), f"#{start + k + 1} {os.path.basename(r.image)[:28]}", fill="black")
            draw.text((x + 3, y + tile + 16), f"given: {r.given_label[:30]}", fill="darkred")
            draw.text((x + 3, y + tile + 30), f"pred:  {r.oof_pred[:24]} ({r.label_quality:.2f})", fill="darkblue")
        path = f"{out_prefix}_p{len(pages) + 1}.jpg"
        sheet.save(path, quality=90)
        pages.append(path)
    return pages


def label_quality(name):
    """5-fold OOF label-quality scores for one component's training images."""
    df, classes = _component_rows(name, "train")
    K = len(classes)
    if K < 2:
        print(f"\n[LabelQuality] {name}: only one class — nothing to score, skipped")
        return None
    X = _load_features("train")[df["row"].to_numpy()]
    y = df["y"].to_numpy()
    groups, linked = _near_dup_groups(list(df["rel"]))
    cv = StratifiedGroupKFold(n_splits=LQ["n_folds"], shuffle=True, random_state=LQ["fold_seed"])
    oof = np.zeros((len(df), K))
    fold_of = np.zeros(len(df), dtype=int)
    for f, (tr, te) in enumerate(cv.split(X, y, groups)):
        sc = StandardScaler().fit(X[tr])
        inner = min(3, int(np.bincount(y[tr], minlength=K)[np.bincount(y[tr], minlength=K) > 0].min()))
        C = (LogisticRegressionCV(Cs=LQ["Cs"], cv=inner, max_iter=5000).fit(sc.transform(X[tr]), y[tr]).C_[0]
             if inner >= 2 else 1.0)
        clf = LogisticRegression(C=C, max_iter=5000).fit(sc.transform(X[tr]), y[tr])
        oof[te] = _predict_full(clf, sc.transform(X[te]), K)
        fold_of[te] = f
    oof = np.clip(oof, 1e-6, 1)
    oof /= oof.sum(1, keepdims=True)

    try:
        from cleanlab.rank import get_label_quality_scores
        from cleanlab.filter import find_label_issues
        quality = get_label_quality_scores(y, oof, method="self_confidence")
        issue = np.zeros(len(df), dtype=bool)
        if K > 1:
            issue = find_label_issues(y, oof, filter_by="prune_by_noise_rate")
        method = "cleanlab (confident learning)"
    except ImportError:
        quality = oof[np.arange(len(df)), y]
        thr = np.array([quality[y == k].mean() if (y == k).any() else 1.0 for k in range(K)])
        issue = (oof.argmax(1) != y) & (oof.max(1) >= thr[oof.argmax(1)])
        method = "self-confidence fallback (pip install cleanlab for confident learning)"

    df["fold"] = fold_of
    df["oof_pred"] = np.array(classes)[oof.argmax(1)]
    df["p_given"] = oof[np.arange(len(df)), y]
    df["label_quality"] = quality
    df["label_issue"] = issue
    df["student_weight"] = np.clip(quality, LQ["weight_floor"], 1.0)
    df = pd.concat([df, _image_quality(list(df["image"]))], axis=1)
    tf_csv = os.path.join(CONFIG_BASE["output_dir"], f"{CONFIG_BASE['experiment_prefix']}_teacher", f"{name}_teacher_train.csv")
    if os.path.exists(tf_csv):                       # cross-check with section 12, if it was run
        t = pd.read_csv(tf_csv)[["image", "seeds_disagreeing", "p_given"]].rename(
            columns={"p_given": "teacher_p_given"})
        df = df.merge(t, on="image", how="left")

    acc = (df["oof_pred"] == df["given_label"]).mean()
    print(f"\n{'=' * 70}\n  LABEL QUALITY — {name}: {len(df)} train images, {K} classes, "
          f"{LQ['n_folds']} folds ({linked} look-alike pairs kept together)\n{'=' * 70}")
    print(f"  out-of-fold probe accuracy : {100 * acc:.2f}%   ({method})")
    print(f"  flagged label issues       : {int(issue.sum())} ({100 * issue.mean():.1f}%)")
    per = df.groupby("given_label").agg(images=("image", "size"), oof_acc=("p_given", lambda s: 0.0),
                                        mean_quality=("label_quality", "mean"), flagged=("label_issue", "sum"))
    per["oof_acc"] = df.assign(ok=df["oof_pred"] == df["given_label"]).groupby("given_label")["ok"].mean()
    print(per.round(3).to_string())
    corr = df[["label_quality", "blur_laplacian_var", "brightness", "contrast"]].corr(method="spearman")["label_quality"]
    print("  Spearman(label quality, image quality): " +
          ", ".join(f"{k} {v:+.2f}" for k, v in corr.drop("label_quality").items()))
    if "seeds_disagreeing" in df:
        print(f"  flagged here AND rejected by >=3 teacher seeds: "
              f"{int((df['label_issue'] & (df['seeds_disagreeing'] >= 3)).sum())}")

    out_dir = LQ["out_dir"]
    os.makedirs(out_dir, exist_ok=True)
    df.drop(columns=["row"]).to_csv(os.path.join(out_dir, f"{name}_label_quality.csv"), index=False)
    rev = df[df["label_issue"]].sort_values("label_quality").copy()
    rev["is_mislabeled (y/n)"] = ""
    rev["correct_label_if_known"] = ""
    rev_csv = os.path.join(out_dir, f"{name}_review.csv")
    keep = ["image", "rel", "given_label", "oof_pred", "label_quality", "p_given", "blur_laplacian_var",
            "is_mislabeled (y/n)", "correct_label_if_known"]
    rev[keep].to_csv(rev_csv, index=False)
    sheets = contact_sheet(rev, os.path.join(out_dir, f"{name}_review_sheet")) if len(rev) else []
    print(f"  review: {rev_csv}" + (f" + {len(sheets)} contact sheet(s)" if sheets else ""))
    return df


def probe_effect(name, lq_df=None):
    """Proof of concept on VALIDATION: same linear probe trained on (A) all labels,
    (B) labels weighted by quality, (C) flagged images removed. Higher val accuracy /
    lower val log-loss for B or C = evidence that cleaning the labels helps."""
    if len(component_class_names(name)) < 2:
        return pd.DataFrame()
    lq_df = lq_df if lq_df is not None else pd.read_csv(os.path.join(LQ["out_dir"], f"{name}_label_quality.csv"))
    tr, classes = _component_rows(name, "train")
    va, _ = _component_rows(name, "val")
    tr = tr.merge(lq_df[["image", "student_weight", "label_issue"]], on="image", how="left")
    K = len(classes)
    Xtr, Xva = _load_features("train")[tr["row"]], _load_features("val")[va["row"]]
    sc = StandardScaler().fit(Xtr)
    Xtr, Xva = sc.transform(Xtr), sc.transform(Xva)
    ytr, yva = tr["y"].to_numpy(), va["y"].to_numpy()
    C = LogisticRegressionCV(Cs=LQ["Cs"], cv=3, max_iter=5000).fit(Xtr, ytr).C_[0]   # chosen on TRAIN, shared by all arms
    keep = ~tr["label_issue"].astype(bool).to_numpy()
    arms = {"A all labels": (np.ones(len(tr), bool), None),
            "B weighted": (np.ones(len(tr), bool), tr["student_weight"].to_numpy()),
            "C flagged removed": (keep, None)}
    rows = []
    for arm, (m, w) in arms.items():
        clf = LogisticRegression(C=C, max_iter=5000).fit(Xtr[m], ytr[m], sample_weight=None if w is None else w[m])
        p = np.clip(_predict_full(clf, Xva, K), 1e-7, 1)
        rows.append({"component": name, "arm": arm, "train_images": int(m.sum()),
                     "val_acc": float((p.argmax(1) == yva).mean()),
                     "val_logloss": float(log_loss(yva, p / p.sum(1, keepdims=True), labels=list(range(K)))),
                     "val_correct": int((p.argmax(1) == yva).sum()), "val_n": len(yva)})
    return pd.DataFrame(rows)


print("Label-quality (5-fold OOF) defined.")


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# 13b. RUN — features (GPU, ~10–20 min, checkpointed) then folds + scoring (CPU, minutes)
#   pip install cleanlab   (once, in this environment) — otherwise a simpler fallback is used
# ═══════════════════════════════════════════════════════════════════════════════
if extract_features(splits=("train", "val"), batch_size=16):
    lq_results, poc = {}, []
    for name in COMPONENTS:
        d = label_quality(name)
        if d is not None:
            lq_results[name] = d
            poc.append(probe_effect(name, d))
    poc = pd.concat(poc, ignore_index=True)
    print("\n" + "=" * 70 + "\n  PROOF OF CONCEPT — linear probe, judged on VALIDATION\n" + "=" * 70)
    print(poc.assign(val_acc=(100 * poc["val_acc"]).round(2), val_logloss=poc["val_logloss"].round(4))
             .to_string(index=False))
    poc.to_csv(os.path.join(LQ["out_dir"], "proof_of_concept_val.csv"), index=False)
    summary = pd.DataFrame([{"component": n, "train_images": len(d), "flagged": int(d["label_issue"].sum()),
                             "oof_probe_acc": round(100 * (d["oof_pred"] == d["given_label"]).mean(), 2),
                             "mean_label_quality": round(d["label_quality"].mean(), 3)}
                            for n, d in lq_results.items()])
    print("\n" + summary.to_string(index=False))
    summary.to_csv(os.path.join(LQ["out_dir"], "label_quality_summary.csv"), index=False)
else:
    print("Features not complete — re-run this cell.")


## 14. v2 methodology on the v1 test composition (216 vs 236)
Cached predictions only (run 11a first). Primary result stays all 236 images.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# 14. v2 METHODOLOGY ON THE v1 TEST COMPOSITION (216 images) — no GPU, no retraining
#   Uses the predictions cached in section 11 (same checkpoints, same loader).
#   original_216 = test images that passed the same quality criteria as training
#   restored_20  = the 20 images removed in v1 (criteria applied without model predictions)
# ═══════════════════════════════════════════════════════════════════════════════
MANIFEST = os.path.join(os.path.dirname(CONFIG_BASE["dataset_root"]), "reports", "manifest.csv")
_man = pd.read_csv(MANIFEST)
RESTORED = set(_man.loc[_man["status"].astype(str).str.startswith("restored"), "rel"])
SEEDS = CONFIG_BASE["multi_seed_list"]


def _tag_restored(df):
    rel = df["true_class"] + "/" + df["image"].map(os.path.basename)
    return df.assign(restored=rel.isin(RESTORED))


def subset_accuracy(df):
    df = _tag_restored(df)
    return {"all_236": df["final_correct"].mean(),
            "original_216": df.loc[~df["restored"], "final_correct"].mean(),
            "restored_20": df.loc[df["restored"], "final_correct"].mean(),
            "routing_216": df.loc[~df["restored"], "routing_correct"].mean()}


check = _tag_restored(split_table("test").assign(final_correct=True))
n_orig, n_rest = int((~check["restored"]).sum()), int(check["restored"].sum())
print(f"Test images: {len(check)} = original {n_orig} + restored {n_rest}")
if (n_orig, n_rest) != (216, 20):
    raise RuntimeError(f"Expected 216 / 20 — check MANIFEST path: {MANIFEST}")

rows = [{"model": f"seed {s}", **subset_accuracy(pipeline_from_probs("test", [s]))} for s in SEEDS]
single = pd.DataFrame(rows).set_index("model")
ens_df = _tag_restored(pipeline_from_probs("test", SEEDS))
ens = subset_accuracy(ens_df)

print("\nPer seed (%):\n" + (single * 100).round(2).to_string())
summary = []
for col in single.columns:
    v = single[col].to_numpy()
    lo, hi = stats.t.interval(0.95, len(v) - 1, loc=v.mean(), scale=stats.sem(v))
    summary.append({"subset": col, "single_mean": 100 * v.mean(), "single_std": 100 * v.std(ddof=1),
                    "ci_low": 100 * lo, "ci_high": 100 * hi, "ensemble": 100 * ens[col]})
summary = pd.DataFrame(summary)
print("\nSingle model (5 seeds) vs 5-seed ensemble (%):\n" + summary.round(2).to_string(index=False))

o = ens_df[~ens_df["restored"]]
per = o.groupby("true_species").agg(images=("image", "size"), routed_ok=("routing_correct", "sum"),
                                    correct=("final_correct", "sum"))
per["end_to_end_acc"] = (per["correct"] / per["images"]).round(4)
r = ens_df[ens_df["restored"]].groupby("true_species")["final_correct"].agg(restored_images="size", restored_correct="sum")
print("\nEnsemble per species on original_216 (+ restored images per species):\n" + per.join(r).fillna(0).to_string())

summary.to_csv(os.path.join(PIPELINE_DIR, "test_216_vs_236_summary.csv"), index=False)
single.to_csv(os.path.join(PIPELINE_DIR, "test_216_vs_236_per_seed.csv"))
print(f"\nSaved to {PIPELINE_DIR}")
